<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h1 style="color: #1e3a8a; border-bottom: 3px solid #1e3a8a; padding-bottom: 8px; margin-bottom: 4px;">PPE Compliance Detection Pipeline</h1>

<h2 style="color: #1e40af; font-weight: 600; margin-top: 6px; margin-bottom: 20px;">Cross-Dataset Evaluation on Construction-Site Imagery</h2>

<table style="border-collapse: collapse; font-size: 14px; margin-bottom: 20px;">
  <tr><td style="padding: 4px 16px 4px 0; color: #475569; font-weight: 600;">Submitters</td><td style="padding: 4px 0;">Raz Albaz (ID 315837658) &middot; Mika Melamed (ID 207488453)</td></tr>
  <tr><td style="padding: 4px 16px 4px 0; color: #475569; font-weight: 600;">Course</td><td style="padding: 4px 0;">Deep Learning Workshop &middot; Dr. Anat Goldstein &middot; M.Sc. Knowledge &amp; Data Engineering, Ariel University</td></tr>
  <tr><td style="padding: 4px 16px 4px 0; color: #475569; font-weight: 600;">Semester</td><td style="padding: 4px 0;">Spring 2026</td></tr>
  <tr><td style="padding: 4px 16px 4px 0; color: #475569; font-weight: 600;">Repository</td><td style="padding: 4px 0;"><code>github.com/Razelbaz1/ppe-compliance-detection</code></td></tr>
</table>

<h3 style="color: #1e40af; margin-top: 18px;">Abstract</h3>

<p style="text-align: justify;">
Construction-site safety depends on workers wearing personal protective equipment (PPE). This project builds
and evaluates a <strong>two-stage deep-learning pipeline</strong> that decides, per worker, whether they
comply with two requirements &mdash; wearing a hard-hat and a high-visibility vest. <strong>Stage&nbsp;1</strong>
is a YOLOv8n detector that localises each worker; <strong>Stage&nbsp;2</strong> is a fine-tuned ResNet50 that
classifies each cropped worker for helmet and vest presence; <strong>Stage&nbsp;3</strong> applies a
deterministic compliance rule. The system is evaluated under a deliberate <strong>Cross-Dataset protocol</strong>:
trained on the Kaggle Construction-Site-Safety dataset and tested both in-domain and on the entirely unseen
Ultralytics Construction-PPE dataset. The two-stage design proves its worth &mdash; the classifier's
cross-dataset AUC drop (~7&nbsp;pp) is about one third of the detector's mAP drop (~20&nbsp;pp), because a
cropped worker varies far less across domains than a full scene. End-to-end, the pipeline returns the correct
compliance verdict for <strong>96%</strong> of detected workers in-domain and <strong>66%</strong>
out-of-domain, where errors are dominated by <em>safe over-flagging</em> (compliant workers wrongly flagged)
rather than dangerous misses. We trace this behaviour to the class-prior shift between domains and outline
domain-adaptation directions for closing the gap.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">1. Introduction</h2>

<h3 style="color: #1e40af; margin-top: 18px;">1.1 The Problem</h3>
<p style="text-align: justify;">
Non-compliance with safety procedures on construction sites endangers human lives.
This project trains a deep learning model on images of construction-site workers to <strong>distinguish, per worker, who is complying with safety procedures and who is not</strong>.
The safety procedures considered in scope are <strong>wearing a hard-hat (helmet)</strong> and <strong>wearing a high-visibility vest</strong>.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">1.2 Project Goal</h3>
<p style="text-align: justify;">
Build a <strong>two-stage pipeline</strong> that decomposes the problem into two well-understood sub-tasks, each solved by an appropriate model:
</p>
<ol>
  <li><strong>Stage 1 &mdash; Person Detection:</strong> locate each worker in the image as a bounding box, using a YOLOv8 detector fine-tuned on construction-site data.</li>
  <li><strong>Stage 2 &mdash; PPE Classification:</strong> crop each detected worker and pass it through a CNN (with transfer learning) that outputs two independent multi-label predictions &mdash; "is wearing helmet?" and "is wearing vest?".</li>
  <li><strong>Stage 3 &mdash; Compliance Logic:</strong> a simple rule on top of Stage 2 outputs &mdash; <code>compliant = helmet &and; vest</code>. Any other combination is flagged as non-compliance with the missing item itemised.</li>
</ol>
<p style="text-align: justify;">
This two-stage decomposition is deliberate: each stage solves a classical, well-bounded task (detection vs. classification) for which strong pretrained models are publicly available &mdash; enabling effective transfer learning even with modestly-sized data.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">1.3 Evaluation Metrics</h3>

<p style="text-align: justify;">
Each stage is a different kind of task, so each is judged by the metric standard for that task, plus one
measure that runs across the whole project (all are defined in detail in Chapter&nbsp;2):
</p>
<ul>
  <li><strong>Stage&nbsp;1 (detection)</strong> &mdash; <strong>mAP@0.5</strong> (and mAP@0.5:0.95), precision
      and recall: the standard way to score how well predicted person boxes match the real ones.</li>
  <li><strong>Stage&nbsp;2 (classification)</strong> &mdash; <strong>AUC</strong> as the headline
      (threshold-independent and robust to the helmet/vest class imbalance), alongside precision, recall and
      accuracy.</li>
  <li><strong>End-to-End</strong> &mdash; <strong>per-worker compliance accuracy</strong>: on the workers the
      detector actually found, did the full pipeline reach the correct compliant / non-compliant verdict.</li>
  <li><strong>Cross-Dataset Gap</strong> &mdash; for every metric above, the drop from the in-domain test set
      to the unseen out-of-domain set. This gap, more than any single number, is what the project sets out to
      measure.</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">1.4 Out of Scope</h3>
<ul>
  <li>Detection of additional PPE items (gloves, goggles, safety boots) &mdash; the datasets support them, but we focus on hard-hat and vest, which are the most safety-critical and the best-represented classes in the data.</li>
  <li>Worker tracking over time (<em>multi-object tracking</em>) &mdash; each frame is treated independently; no video-sequence reasoning.</li>
  <li>Worker identification &mdash; workers are treated as anonymous objects.</li>
  <li>Production deployment &mdash; real-time camera streaming, model optimization for edge devices, MLOps. Outside the scope of an academic course project.</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">1.5 Notebook Structure</h3>
<p>The notebook is organised by chapters. Each chapter contains a brief theoretical explanation, the code that implements it, and a summary of what we learned from it:</p>
<ol>
  <li>Introduction and project goals (current chapter)</li>
  <li>Theoretical background &mdash; Object Detection, Transfer Learning, IoU, evaluation metrics</li>
  <li>Environment setup and imports</li>
  <li><strong>Data split &mdash; Cross-Dataset Strategy</strong></li>
  <li>Exploratory Data Analysis (EDA)</li>
  <li>Deduplication and leakage check (pHash)</li>
  <li>Data preparation for Stage 1</li>
  <li><strong>Stage 1</strong> &mdash; YOLOv8 training for person detection</li>
  <li>Stage 1 evaluation</li>
  <li>Worker-crop dataset construction for Stage 2</li>
  <li><strong>Stage 2</strong> &mdash; CNN multi-label classifier training</li>
  <li>Stage 2 evaluation</li>
  <li><strong>Stage 3</strong> &mdash; Compliance Logic</li>
  <li>End-to-End evaluation on full images</li>
  <li>Discussion, limitations, and future work</li>
  <li>Summary and conclusions</li>
  <li>References</li>
</ol>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">2. Theoretical Background</h2>

<p style="text-align: justify;">
Before we start coding the pipeline, this chapter reviews the concepts that will appear in the code in later chapters.
Each concept is presented in a <strong>three-layer format</strong>: <em>intuition</em> &middot;
<em>mathematical formulation</em> &middot; <em>why it matters in this project</em>.
</p>

<p>The chapter is split into two worlds that correspond to the two stages of the pipeline:</p>
<ul>
  <li><strong>2.1&ndash;2.5 &mdash; The detection world:</strong> how objects are located inside an image &mdash;
      everything needed to understand Stage 1.</li>
  <li><strong>2.6&ndash;2.9 &mdash; The classification world:</strong> Transfer Learning, multi-label classification,
      the loss function, and augmentation &mdash; everything needed to understand Stage 2.</li>
</ul>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h3 style="color: #1e40af; margin-top: 18px;">2.1 Object Detection &mdash; how it differs from regular classification</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> <strong>Classification</strong> says &quot;this image contains a cat&quot; &mdash; one label for the whole image. <strong>Detection</strong> says &quot;<em>here</em> in the image is a cat, <em>here</em> is a dog, and here are other objects&quot; &mdash; every object gets both a class and a position.</p>
<p><strong style="color:#1e40af;">Formally.</strong> Classification model: <code>f(image) &rarr; vector of K probabilities</code> (one per class).<br>Detection model: <code>f(image) &rarr; list of {bbox, class, confidence}</code> &mdash; one entry per detected object. The number of outputs is <em>not fixed</em>: one image may contain 0 objects or 50.</p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> A construction-site image can contain one worker or twenty. Classification alone is not enough &mdash; we need a detector that gives us a separate bbox for each worker, so the PPE classifier can be applied to each worker independently.</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.2 Bounding Box (bbox) &mdash; how an object is represented</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> A tight rectangle that wraps an object in the image. The detector has two parallel jobs: <em>where</em> the object is (bbox) and <em>what</em> it is (class).</p>
<p><strong style="color:#1e40af;">Formally.</strong> Represented by 4 numbers. Three common formats appear in code:<table style='border-collapse:collapse;font-size:13px;margin-top:6px;'><tr><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Format</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Fields</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Where it's used</th></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;"><code>XYXY</code></td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">(x_min, y_min, x_max, y_max)</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">Pascal VOC, OpenCV</td></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;"><code>XYWH</code></td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">(x, y, width, height)</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">COCO</td></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;"><code>YOLO</code></td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">(cx, cy, w, h) &mdash; normalized to [0, 1]</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;"><strong>Our format</strong></td></tr></table></p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> Each <code>.txt</code> file in <code>datasets/.../labels/</code> contains one bbox per line in YOLO format. Example from the actual data:<pre style='background:#f8fafc;padding:8px;border-radius:4px;font-size:13px;'>0 0.523 0.412 0.180 0.350</pre><strong>Reading it:</strong> class 0 (helmet), bbox center at (52.3%, 41.2%) of the image, width 18% and height 35% of the image dimensions. This is what YOLO reads at training time and what it predicts at inference time.</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.3 IoU &mdash; Intersection over Union</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> A metric that captures &quot;how much do two rectangles overlap&quot;. A value of 1 means they overlap perfectly; 0 means they do not touch at all.</p>
<p><strong style="color:#1e40af;">Formally.</strong> <code>IoU(A, B) = |A &cap; B| / |A &cup; B|</code><br><strong>Numeric example:</strong> predicted bbox = (10, 10, 50, 50), ground-truth bbox = (20, 20, 60, 60).<br>&bull; Intersection area = 30 &times; 30 = <strong>900</strong><br>&bull; Union area = 1600 + 1600 &minus; 900 = <strong>2300</strong><br>&bull; IoU = 900 / 2300 &asymp; <strong>0.39</strong> (a partial overlap).</p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> IoU shows up in three places in our project:<table style='border-collapse:collapse;font-size:13px;margin-top:6px;'><tr><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Where</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Role</th></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">During YOLO training</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">YOLO computes IoU between its prediction and ground truth as part of the loss</td></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">During evaluation</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">A prediction counts as &quot;correct&quot; if IoU with ground truth &ge; 0.5 (the threshold for mAP@0.5)</td></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">In our <code>build_crop_dataset.py</code></td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">We compute IoU <em>by hand</em> to decide whether a person crop is labelled as &quot;wearing a helmet&quot; &mdash; by checking whether a helmet bbox overlaps that person's bbox</td></tr></table></p>

<h3 style="color: #1e40af; margin-top: 18px;">2.4 YOLO &mdash; the detector architecture we use</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> <strong>Y</strong>ou <strong>O</strong>nly <strong>L</strong>ook <strong>O</strong>nce. Unlike R-CNN, which looks at the image twice (first to find candidate regions, then to classify them), YOLO performs the entire detection in a <em>single forward pass</em> &mdash; which makes it very fast.</p>
<p><strong style="color:#1e40af;">Formally.</strong> The image passes through a CNN that produces feature maps at several resolutions (FPN). Each grid cell is &quot;responsible&quot; for objects whose center falls inside it, and predicts:<ul><li>B bounding boxes (position + size)</li><li>A probability distribution over K classes</li><li>A confidence score that captures &quot;how sure am I there is an object here&quot;</li></ul><strong>YOLOv8 is anchor-free:</strong> unlike earlier YOLO versions, it does not rely on pre-defined &quot;anchor boxes&quot; &mdash; it predicts the bbox directly.</p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> We adopt the YOLO architecture because our data is well-suited to it: a single-class detection task (person) with a moderate number of training images, where pretrained YOLO weights on COCO provide a strong starting point.</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.5 mAP, Precision, Recall &mdash; how detectors are measured</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> <strong>Precision</strong> = of all the detections I made, how many were correct? &quot;How often do I lie?&quot;<br><strong>Recall</strong> = of all the objects that were truly in the image, how many did I find? &quot;How often am I lazy?&quot;<br><strong>mAP</strong> = a single score that combines both.</p>
<p><strong style="color:#1e40af;">Formally.</strong> Every prediction is sorted into one of four categories based on IoU against ground truth:<ul><li><strong>TP</strong> (True Positive): IoU &ge; 0.5 AND the class is correct</li><li><strong>FP</strong> (False Positive): a detection that does not match any real object</li><li><strong>FN</strong> (False Negative): a real object that was not detected</li></ul><code>Precision = TP / (TP + FP)</code><br><code>Recall = TP / (TP + FN)</code><br><code>AP = area under the Precision-Recall curve</code><br><code>mAP = mean AP averaged over all classes</code></p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> YOLO reports <strong>two</strong> mAP variants:<ul><li><code>mAP@0.5</code> &mdash; IoU threshold of 0.5 (lenient).</li><li><code>mAP@0.5:0.95</code> &mdash; average across 10 thresholds (0.5, 0.55, &hellip;, 0.95). Much stricter.</li></ul>In our project: <em>low Precision</em> means the model often shouts &quot;I see a worker!&quot; when none is there; <em>low Recall</em> means real workers are missed. For PPE compliance, <strong>Recall is the more important of the two</strong> &mdash; missing a worker means missing a potential non-compliance event.</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h3 style="color: #1e40af; margin-top: 18px;">2.6 Transfer Learning &mdash; building on pretrained models</h3>

<p><strong style="color:#1e40af;">Intuition.</strong>
Rather than training a CNN from scratch on our modest dataset (~12K crops),
we start from a model already trained on a massive dataset (ImageNet &mdash; 1.4M images, 1000 classes)
and leverage the representations it has already learned &mdash; edges, textures, shapes &mdash; which transfer
well to our domain.
</p>

<p><strong style="color:#1e40af;">Formally.</strong>
Two usage modes:
<ul>
  <li><strong>Feature Extraction:</strong> freeze the backbone (<code>conv_base.trainable = False</code>) and train only a fresh classifier head (a new Dense layer).</li>
  <li><strong>Fine-Tuning:</strong> unfreeze the top layers of the backbone and continue training with a <em>very low</em> learning rate (1e-5) so the pretrained weights are not destroyed.</li>
</ul>
</p>

<p><strong style="color:#1e40af;">Why we use it here.</strong>
Transfer Learning is the natural choice for this project because, as demonstrated in the course lecture on Transfer Learning (Unit 8 &mdash; Cats vs Dogs), training from scratch on a small dataset reached only ~66% accuracy, while feature extraction reached ~83% and fine-tuning reached ~97% &mdash; with only 2,000 training images.
With ~12K crops in our Stage 2 dataset, the same approach is the only practical way to reach competitive accuracy: training the CNN from scratch would leave a large fraction of the model's potential on the table.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.7 Multi-Label Classification &mdash; multiple labels per example</h3>
<p><strong style="color:#1e40af;">Intuition.</strong> <strong>Multi-class</strong> asks &quot;what is this?&quot; &mdash; <em>one</em> of K choices (cat <em>or</em> dog <em>or</em> bird).<br><strong>Multi-label</strong> asks &quot;which attributes does it have?&quot; &mdash; <em>any subset</em> of K attributes (wearing helmet <em>and</em> wearing vest).</p>
<p><strong style="color:#1e40af;">Formally.</strong> <strong>The critical architectural difference:</strong><table style='border-collapse:collapse;font-size:13.5px;margin-top:6px;'><tr><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;"></th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Output layer</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Activation</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Loss</th><th style="padding: 4px 10px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Sum of probabilities</th></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">Multi-class</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">Dense(K)</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">softmax</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">categorical_crossentropy</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">= 1</td></tr><tr><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">Multi-label</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">Dense(K)</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">sigmoid</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">binary_crossentropy</td><td style="padding: 4px 10px; border: 1px solid #cbd5e1;">not necessarily = 1</td></tr></table><p>In our project: K=2 (helmet, vest), two independent sigmoid neurons. Example output: <code>[0.87, 0.12]</code> = &quot;87% confidence wearing a helmet, 12% confidence wearing a vest&quot;.</p></p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> If we mistakenly used softmax with 2 heads, we would force helmet and vest to <em>compete</em> &mdash; raising the helmet probability would lower the vest probability. This does not match reality: a worker can wear both. Independent sigmoids allow independent decisions.</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.8 BCE Loss &mdash; Binary Cross-Entropy</h3>

<div style="background: #f0fdf4; border-left: 3px solid #16a34a; padding: 6px 12px; margin: 8px 0; border-radius: 4px 0 0 4px; font-size: 13.5px; color: #14532d;">
<strong>&#9989; Covered in the course:</strong> Unit 4 (NN Optimization) and Unit 6 (Keras &mdash; IMDB classification).
</div>

<p><strong style="color:#1e40af;">Intuition.</strong> A metric that says &quot;how far is my prediction from the ground truth&quot; for a binary problem. The more confidently wrong the model is, the higher the loss; the more correct it is, the closer the loss is to 0.</p>
<p><strong style="color:#1e40af;">Formally.</strong> For one example and one label:<br><code>BCE(y, &ycirc;) = &minus;[y &middot; log(&ycirc;) + (1&minus;y) &middot; log(1&minus;&ycirc;)]</code><br>where <code>y &isin; {0, 1}</code> (the ground truth) and <code>&ycirc; &isin; [0, 1]</code> (sigmoid output).<br><br><strong>For the multi-label case:</strong> BCE is summed over all K labels; in Keras, <code>binary_crossentropy</code> automatically averages.<br><br><strong>Numeric example:</strong> y=1, &ycirc;=0.9 &rArr; BCE = &minus;log(0.9) &asymp; 0.105 (low, good).<br>y=1, &ycirc;=0.1 &rArr; BCE = &minus;log(0.1) &asymp; 2.30 (high &mdash; the model is penalized for confident wrongness).</p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> BCE is the standard loss for Stage 2 in our project &mdash; <code>model.compile(loss='binary_crossentropy', metrics=['accuracy', 'AUC'])</code>. What makes it appropriate: it is <em>asymmetric</em> &mdash; it punishes very-wrong predictions much more than slightly-wrong ones. This pushes the model to be honest rather than declare &quot;100% sure&quot; without justification.</p>

<h3 style="color: #1e40af; margin-top: 18px;">2.9 Data Augmentation &mdash; artificially expanding the dataset</h3>

<div style="background: #f0fdf4; border-left: 3px solid #16a34a; padding: 6px 12px; margin: 8px 0; border-radius: 4px 0 0 4px; font-size: 13.5px; color: #14532d;">
<strong>&#9989; Covered in the course:</strong> Unit 8 (Cats vs Dogs) &mdash; demonstrated to push back overfitting from epoch 5 to epoch 30+.
</div>

<p><strong style="color:#1e40af;">Intuition.</strong> &quot;Invent new images&quot; from the existing ones by applying random transformations (flip, rotation, brightness change, crop). Each epoch the network sees slightly different versions of the same image &mdash; as if the dataset were several times larger.</p>
<p><strong style="color:#1e40af;">Formally.</strong> <strong>Common techniques we will use:</strong><ul><li><strong>Geometric:</strong> <code>RandomFlip('horizontal')</code> (important!), <code>RandomRotation(0.05)</code>, <code>RandomZoom(0.1)</code></li><li><strong>Photometric:</strong> <code>RandomBrightness(0.2)</code>, <code>RandomContrast(0.2)</code> &mdash; essential because lighting varies a lot across construction sites</li></ul><strong>What we will <em>not</em> use:</strong> vertical flip &mdash; an upside-down worker is not a realistic scenario and would teach the network that a back = a head.</p>
<p><strong style="color:#1e40af;">Why it matters here.</strong> In Stage 1, YOLO performs its own built-in augmentation (mosaic, mixup, copy-paste) &mdash; we don't need to do anything. In Stage 2 (CNN), we will build an augmentation pipeline in Keras as preprocessing layers. Without augmentation, with only ~12K crops, there is a serious risk of overfitting after 5&ndash;10 epochs.</p>

<div style="background: #eff6ff; border: 1px solid #bfdbfe; border-radius: 6px; padding: 10px 14px; margin-top: 20px; color: #1e3a8a;">
  <strong>&#9989; Chapter summary.</strong> We are now equipped with the concepts that will appear in every later chapter &mdash;
  Detection (bbox, IoU, mAP, YOLO) and Classification (Transfer Learning, multi-label, BCE, augmentation).
</div>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">3. Environment Setup</h2>

<p style="text-align: justify;">
The two code cells below prepare the runtime so the rest of the notebook can run:
</p>

<ul>
  <li><strong>3.1 &mdash; Environment detection &amp; paths.</strong> Detects whether the notebook is running locally, on a remote workstation (SSH), or on Google Colab, and resolves <code>REPO_ROOT</code> and <code>DATA_ROOT</code> accordingly.</li>
  <li><strong>3.2 &mdash; Imports, seeds, and GPU check.</strong> Imports the required libraries, seeds <code>random</code> / <code>numpy</code> / <code>tensorflow</code> / <code>torch</code> with <code>SEED = 42</code>, and reports which GPU (if any) is available.</li>
</ul>

<p style="text-align: justify;">
Setup instructions &mdash; installation, dataset download, runtime auto-detection, the laptop-to-workstation workflow, and reproducibility caveats &mdash; live in the <strong><a href="../README.md#5-runtime-environments--workflow">Runtime Environments &amp; Workflow</a></strong> section of the README, so this chapter stays focused on the notebook itself.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 3.1 - Environment detection and path resolution
# =============================================================================
import os
import sys
import platform
from pathlib import Path

# --- 1. Detect runtime environment -------------------------------------------
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_SSH = bool(os.environ.get("SSH_CONNECTION") or os.environ.get("SSH_CLIENT"))
OS_NAME = platform.system()  # 'Windows', 'Linux', 'Darwin'

def _env_label() -> str:
    if IS_COLAB:
        return "Google Colab"
    if IS_SSH:
        return f"Linux SSH ({platform.node()})"
    return f"{OS_NAME} (Local)"

print(f"Environment   : {_env_label()}")

# --- 2. Resolve REPO_ROOT and DATA_ROOT --------------------------------------
if IS_COLAB:
    # Mount Drive (idempotent). Fallback to local-style search if it fails
    # (e.g. Colab Local Runtime where Drive is not exposed).
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception as e:
        print(f"  WARNING: Drive mount failed ({e}). Treating as local runtime.")
        IS_COLAB = False

if IS_COLAB:
    # Clone the project repo into /content/ if not present
    REPO_ROOT = Path("/content/ppe-compliance-detection")
    if not REPO_ROOT.exists():
        os.system(
            "git clone -q https://github.com/Razelbaz1/ppe-compliance-detection.git "
            f"{REPO_ROOT}"
        )

    DATA_ROOT_DEFAULT = Path("/content/drive/MyDrive/Deep_Learning_course/datasets")
    DATA_ROOT = Path(os.environ.get("PPE_DATA_ROOT", DATA_ROOT_DEFAULT))
else:
    # When launched from the notebooks/ folder, REPO_ROOT is one level up
    nb_dir = Path.cwd().resolve()
    REPO_ROOT = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir

    # Search candidates in priority order
    env_override = os.environ.get("PPE_DATA_ROOT")
    home = Path.home()

    candidates: list = [
        Path(env_override) if env_override else None,
        REPO_ROOT.parent / "datasets",
        REPO_ROOT / "datasets",
        home / "datasets",
        home / "ppe-data" / "datasets",
    ]

    # Linux / SSH-typical shared locations
    if OS_NAME == "Linux":
        candidates += [
            Path("/data/Deep_Learning_course/datasets"),
            Path("/mnt/data/Deep_Learning_course/datasets"),
        ]

    # Windows-specific locations
    if OS_NAME == "Windows":
        candidates += [
            Path("G:/My Drive/Deep_Learning_course/datasets"),
            *list(home.glob("OneDrive*/Desktop/**/Deep_Learning_course/datasets")),
        ]

    candidates = [c for c in candidates if c is not None]

    DATA_ROOT = next(
        (c for c in candidates if (c / "construction-ppe").exists()),
        None,
    )
    if DATA_ROOT is None:
        hint = (
            "$env:PPE_DATA_ROOT = 'C:\\path\\to\\datasets'  # PowerShell"
            if OS_NAME == "Windows"
            else "export PPE_DATA_ROOT=$HOME/path/to/datasets       # bash"
        )
        raise FileNotFoundError(
            "Could not locate the datasets folder.\n"
            "Set PPE_DATA_ROOT to point at the folder containing "
            "'construction-ppe/' and 'kaggle-css/'. Example:\n"
            f"  {hint}"
        )

# --- 3. Derived paths --------------------------------------------------------
ULTRALYTICS_DIR = DATA_ROOT / "construction-ppe"
KAGGLE_DIR      = DATA_ROOT / "kaggle-css"
RESULTS_DIR     = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- 4. Make src/ importable from the notebook -------------------------------
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- 5. Summary --------------------------------------------------------------
def _ok(p: Path) -> str:
    return "OK " if p.exists() else "MISSING"

print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"DATA_ROOT     : {DATA_ROOT}")
print(f"  Ultralytics : {_ok(ULTRALYTICS_DIR)}  -> {ULTRALYTICS_DIR}")
print(f"  Kaggle CSS  : {_ok(KAGGLE_DIR)}  -> {KAGGLE_DIR}")
print(f"RESULTS_DIR   : {RESULTS_DIR}")


In [ ]:
# =============================================================================
# Chapter 3.2 - Imports, seeding, and GPU detection
# =============================================================================
import os
import random
import warnings

# Silence TF/Keras log noise BEFORE importing tensorflow
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# --- Lightweight imports ------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 1. Seeding (must come before any DL framework use) ----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- 2. PyTorch — required from Chapter 7 (YOLO) ------------------------------
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- 3. TensorFlow — required from Chapter 10 (Stage 2 classifier) ------------
# Wrapped in try/except so the notebook still runs through EDA chapters even
# before TF is installed.
try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    HAS_TF = True
except ImportError:
    tf = None
    HAS_TF = False

# --- 4. Library versions -----------------------------------------------------
print("Library versions")
print(f"  numpy        {np.__version__}")
print(f"  pandas       {pd.__version__}")
print(f"  matplotlib   {plt.matplotlib.__version__}")
print(f"  torch        {torch.__version__}")
print(f"  tensorflow   {tf.__version__ if HAS_TF else 'NOT INSTALLED (required from Chapter 10)'}")
print()

# --- 5. GPU detection --------------------------------------------------------
print(f"PyTorch CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - Device : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  - Memory : {total_gb:.1f} GB")
else:
    print("  - CPU only. YOLO training will be slow. Recommended:")
    print("      * Google Colab (free T4 GPU)")
    print("      * Remote machine via SSH with NVIDIA GPU")

if HAS_TF:
    tf_gpus = tf.config.list_physical_devices("GPU")
    print(f"TensorFlow GPUs detected : {len(tf_gpus)}")
    for g in tf_gpus:
        print(f"  - {g.name}")
else:
    print()
    print("  NOTE: TensorFlow is not installed yet. The notebook will run")
    print("  through chapters 4-9 without it; install before Chapter 10:")
    print("      pip install tensorflow>=2.15.0")

print(f"\nRandom seed set to {SEED}.")


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">4. Data Split &mdash; Cross-Dataset Strategy</h2>

<h3 style="color: #1e40af; margin-top: 18px;">4.1 The approach: Cross-Dataset Evaluation</h3>

<p style="text-align: justify;">
Our two datasets &mdash; Ultralytics Construction-PPE and Kaggle CSS &mdash; differ noticeably from each other: different sites, different photography style, and in particular very different compliance distributions (Ultralytics ~61% compliant, Kaggle ~20% compliant).
Rather than merging them into a standard train/val/test split (the common, less challenging approach), we adopt an academic approach called <strong>Cross-Dataset Evaluation</strong> &mdash; the model trains on one dataset entirely and is then evaluated on a different dataset that it has never seen during training.
</p>

<p>
The <strong>project narrative</strong>: <em>"the model succeeds in both worlds"</em> &mdash; both on the domain it has learned (in-domain) and on a completely new domain (out-of-domain).
</p>

<h3 style="color: #1e40af; margin-top: 18px;">4.2 The four splits</h3>

<table style="border-collapse: collapse; font-size: 13.5px; margin-top: 6px;">
  <thead><tr>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Split</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Source</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Size</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Role</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Train</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Kaggle CSS &mdash; train</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">2,605</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Model learns here (YOLO and CNN)</td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Validation</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Kaggle CSS &mdash; val</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">114</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Early stopping, hyperparameter selection</td>
    </tr>
    <tr style="background:#fef3c7;">
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Test In-Domain</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Kaggle CSS &mdash; test</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">80</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">How well does the model do on its known domain?</td>
    </tr>
    <tr style="background:#fef3c7;">
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Test Out-of-Domain</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Ultralytics &mdash; entire dataset</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">1,414</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">How well does the model generalize to a new domain?</td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Total</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">&mdash;</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>4,213</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">&mdash;</td>
    </tr>
  </tbody>
</table>

<p style="font-size: 13px; color: #64748b;">
Sizes are reported <strong>after</strong> pHash deduplication of 4 duplicates (detailed in Chapter 6).
Stage 1 and Stage 2 share the same split &mdash; a person crop from Train cannot appear in val or test, since it is derived from a whole image that has already been allocated.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">5. Exploratory Data Analysis (EDA)</h2>

<p style="text-align: justify;">
In this chapter we explore the Kaggle CSS dataset, which we chose as the training/validation domain in Chapter 4.
The goal is to understand what the model will be learning from: how many images and bounding boxes we have,
which classes are present and in what proportions, how many objects appear per image, and what a few representative images actually look like.
</p>

<p style="text-align: justify;">
Following the principle established in Chapter 4, EDA is performed on the
<strong>train + validation splits only</strong>. The test sets (Kaggle CSS test, and the entire Ultralytics dataset)
are not touched here &mdash; they remain unseen until evaluation.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 5.2 - Load Kaggle CSS labels (train + val) into DataFrames
# =============================================================================
# Each YOLO-format label file contains one bbox per line:
#   <class_id> <cx> <cy> <w> <h>
# where coordinates are normalized to [0, 1].

KAGGLE_CSS_CLASSES = [
    "Hardhat", "Mask", "NO-Hardhat", "NO-Mask", "NO-Safety Vest",
    "Person", "Safety Cone", "Safety Vest", "machinery", "vehicle",
]


def load_split_labels(split_dir: Path, class_names: list) -> pd.DataFrame:
    """Parse every .txt label file under <split_dir>/labels into a DataFrame."""
    records = []
    labels_dir = split_dir / "labels"
    for txt_file in sorted(labels_dir.glob("*.txt")):
        with open(txt_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                records.append({
                    "image":      txt_file.stem,
                    "split":      split_dir.name,
                    "class_id":   cls_id,
                    "class_name": class_names[cls_id],
                    "cx":         float(parts[1]),
                    "cy":         float(parts[2]),
                    "w":          float(parts[3]),
                    "h":          float(parts[4]),
                })
    return pd.DataFrame(records)


CSS_DATA = KAGGLE_DIR / "css-data"
train_df = load_split_labels(CSS_DATA / "train", KAGGLE_CSS_CLASSES)
val_df   = load_split_labels(CSS_DATA / "valid", KAGGLE_CSS_CLASSES)

print(f"Train : {train_df['image'].nunique():>5,} images   {len(train_df):>6,} bboxes")
print(f"Val   : {val_df['image'].nunique():>5,} images   {len(val_df):>6,} bboxes")
print(f"Test  : NOT LOADED (kept unseen until evaluation)")

train_df.head()


In [ ]:
# =============================================================================
# Chapter 5.3 - Basic statistics on EDA scope (train + val)
# =============================================================================
eda_df = pd.concat([train_df, val_df], ignore_index=True)

# Compare physical image count vs images that actually contain bboxes.
# Empty label files correspond to "background" images with no objects.
def count_label_files(split_dir: Path) -> int:
    return sum(1 for _ in (split_dir / "labels").glob("*.txt"))

total_train = count_label_files(CSS_DATA / "train")
total_val   = count_label_files(CSS_DATA / "valid")
total_imgs  = total_train + total_val

n_images_with_bbox = eda_df["image"].nunique()
n_empty            = total_imgs - n_images_with_bbox
n_bboxes           = len(eda_df)

print(f"Total images in EDA scope (train + val) : {total_imgs:,}")
print(f"  - Images with at least one bbox       : {n_images_with_bbox:,}")
print(f"  - Empty label files (no objects)      : {n_empty:,}")
print(f"Total bounding boxes                    : {n_bboxes:,}")
print(f"Average bboxes per non-empty image      : {n_bboxes / n_images_with_bbox:.2f}")
print()
print("Bboxes per class:")
counts = eda_df["class_name"].value_counts()
for cls, cnt in counts.items():
    pct = 100 * cnt / n_bboxes
    print(f"  {cls:<18s} {cnt:>6,}  ({pct:5.1f}%)")


In [ ]:
# =============================================================================
# Chapter 5.4 - Class distribution (Kaggle CSS train + val)
# =============================================================================
# The classes relevant to our PPE compliance pipeline are highlighted in blue.
# Other classes (Mask, Safety Cone, vehicle, machinery) are present in the raw
# data but not used by our model.

PROJECT_CLASSES = {"Person", "Hardhat", "Safety Vest", "NO-Hardhat", "NO-Safety Vest"}
class_counts = eda_df["class_name"].value_counts()

colors = [
    "#1e40af" if c in PROJECT_CLASSES else "#cbd5e1"
    for c in class_counts.index
]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(class_counts.index, class_counts.values, color=colors,
              edgecolor="white", linewidth=0.5)

# Annotate each bar with its count
for bar, cnt in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{cnt:,}", ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Class")
ax.set_ylabel("Number of bounding boxes")
ax.set_title("Class distribution - Kaggle CSS train + val "
             "(blue = used by our pipeline)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<p style="text-align: justify;">
The chart reveals two characteristics of the data that will directly shape the training pipeline:
</p>

<ul>
  <li><strong>Person is by far the dominant class</strong> (about 26% of all bboxes). This is good news for Stage 1 &mdash; the detector will see ~9,700 worker examples to learn from.</li>
  <li><strong>Roughly 40% of the bboxes belong to classes we will not use</strong> (Mask, Safety Cone, machinery, vehicle, NO-Mask). These will be filtered out in Chapter 7 when we build a Person-only label set for YOLO &mdash; what remains is a cleaner signal for the model.</li>
  <li><strong>The PPE classes are unbalanced and asymmetric.</strong> Hardhat (3,224) outnumbers NO-Hardhat (2,386), but Safety Vest (3,074) is outnumbered by NO-Safety Vest (4,068) &mdash; meaning the dataset shows more workers wearing helmets than not, but more workers <em>without</em> vests than with them. Stage 2 will need class-aware training (class weights or balanced sampling) to avoid biasing toward the majority decision.</li>
</ul>

</div>

In [ ]:
# =============================================================================
# Chapter 5.5 - Distribution of bounding boxes per image
# =============================================================================
bboxes_per_image = eda_df.groupby("image").size()

print(f"Bboxes per image - min : {bboxes_per_image.min()}")
print(f"                  - p25 : {bboxes_per_image.quantile(0.25):.0f}")
print(f"                  - median: {bboxes_per_image.median():.0f}")
print(f"                  - p75 : {bboxes_per_image.quantile(0.75):.0f}")
print(f"                  - max : {bboxes_per_image.max()}")
print(f"                  - mean: {bboxes_per_image.mean():.2f}")

fig, ax = plt.subplots(figsize=(10, 4.5))
bins = range(0, int(bboxes_per_image.max()) + 2)
ax.hist(bboxes_per_image, bins=bins, color="#1e40af", edgecolor="white", linewidth=0.5)
ax.axvline(bboxes_per_image.mean(), color="#dc2626", linestyle="--",
           linewidth=1.5, label=f"mean = {bboxes_per_image.mean():.1f}")
ax.set_xlabel("Number of bounding boxes in image")
ax.set_ylabel("Number of images")
ax.set_title("How many objects appear in a single image?")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# Chapter 5.6 - Sample images with bounding boxes overlaid
# =============================================================================
# Colour code: only classes relevant to our pipeline are drawn in colour;
# others (Mask, Safety Cone, ...) are drawn in grey for context.

CLASS_COLORS = {
    "Person":         "#3b82f6",
    "Hardhat":        "#16a34a",
    "Safety Vest":    "#f59e0b",
    "NO-Hardhat":     "#dc2626",
    "NO-Safety Vest": "#dc2626",
}
DEFAULT_COLOR = "#cbd5e1"


def find_image_file(images_dir: Path, stem: str) -> Path | None:
    for ext in (".jpg", ".jpeg", ".png"):
        candidate = images_dir / (stem + ext)
        if candidate.exists():
            return candidate
    return None


def draw_image_with_bboxes(ax, image_path: Path, bboxes: pd.DataFrame):
    img = Image.open(image_path).convert("RGB")
    W, H = img.size
    ax.imshow(img)
    for _, row in bboxes.iterrows():
        x = (row["cx"] - row["w"] / 2) * W
        y = (row["cy"] - row["h"] / 2) * H
        w = row["w"] * W
        h = row["h"] * H
        color = CLASS_COLORS.get(row["class_name"], DEFAULT_COLOR)
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
    ax.axis("off")


# Pick 6 random images from train, seeded for reproducibility
sample_images = (
    train_df["image"]
    .drop_duplicates()
    .sample(n=6, random_state=SEED)
    .tolist()
)
train_images_dir = CSS_DATA / "train" / "images"

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_name in zip(axes.flat, sample_images):
    img_file = find_image_file(train_images_dir, img_name)
    if img_file is None:
        ax.text(0.5, 0.5, f"Image file missing\n{img_name[:30]}...",
                ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        continue
    img_bboxes = train_df[train_df["image"] == img_name]
    draw_image_with_bboxes(ax, img_file, img_bboxes)

plt.suptitle("Sample images from Kaggle CSS train, with bounding boxes",
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">5.7 EDA summary &mdash; key takeaways</h3>

<p style="text-align: justify;">
The Kaggle CSS train+val data contains <strong>2,719 images</strong> (2,605 train + 114 val), of which <strong>16 are "background" images with empty label files</strong> (no objects annotated). These will still be useful at training time as negative examples for Stage 1.
</p>

<p style="text-align: justify;">
Across all the data, there are <strong>37,592 bounding boxes</strong>, with an average of about <strong>14 objects per image</strong> (median 11, max 87) &mdash; the scenes are dense and will be a real challenge for the detector.
</p>

<p style="text-align: justify;">
Of the ten classes present, our pipeline uses five (Person, Hardhat, Safety Vest, NO-Hardhat, NO-Safety Vest); the other five (Mask, Safety Cone, machinery, vehicle, NO-Mask) make up about 40% of the bboxes and will be filtered out in Chapter 7.
</p>

<p style="text-align: justify;">
Among the PPE-related classes, the dataset is asymmetric: workers are more often shown <em>with</em> a helmet than without, but more often <em>without</em> a vest than with one. This asymmetry will drive design choices in Stage 2 &mdash; class weights, balanced sampling, and aggressive augmentation &mdash; to avoid a model that simply learns to predict the majority answer.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">6. Deduplication &amp; leakage check</h2>

<p style="text-align: justify;">
As part of preparing the data, a perceptual-hash (pHash) deduplication check was run on all 4,217 images
across both datasets, in every train/val/test combination. The check identified <strong>4 duplicate images</strong>
between the two datasets' test sets, which were removed &mdash; leaving <strong>4,213 images</strong> in total.
The verification script lives at <code>src/utils/deduplicate.py</code> and can be re-run at any time.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">7. Data preparation for Stage 1</h2>

<p style="text-align: justify;">
Stage 1 is a single-class person detector, so the multi-class label files of both datasets need to be filtered:
keep only the <strong>Person</strong> bounding boxes, drop everything else, and rewrite the class id to <code>0</code> (since the YOLO model will see only one class).
The filtered data is written to a new directory tree under <code>DATA_ROOT/processed_stage1/</code>,
together with a <code>data.yaml</code> file that points YOLOv8 at the train/val/test splits we defined in Chapter&nbsp;4.
</p>

<p style="text-align: justify; font-size: 13px; color: #475569;">
Note on class ids: in the source data, Person is class <code>5</code> in Kaggle CSS and class <code>6</code> in Ultralytics. After filtering both become class <code>0</code> in the processed labels.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 7.2 - Configuration and the filter-and-link helper
# =============================================================================
import shutil

# Class id of "Person" in each source dataset
KAGGLE_PERSON_CLASS      = 5
ULTRALYTICS_PERSON_CLASS = 6

# Source-data roots (defined here so chapter 7 is independent of chapter 5)
CSS_DATA       = KAGGLE_DIR / "css-data"
PROCESSED_ROOT = DATA_ROOT / "processed_stage1"


def link_or_copy(src: Path, dst: Path) -> None:
    """Hard-link the file if possible (fast, no extra disk), otherwise copy."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except (OSError, NotImplementedError):
        shutil.copy2(src, dst)


def filter_split(
    src_images_dir: Path,
    src_labels_dir: Path,
    dst_dir: Path,
    person_class_id: int,
) -> dict:
    """Filter one split: keep only Person bboxes, link images, write labels.

    Returns a small dict of counts for reporting.
    """
    dst_images = dst_dir / "images"
    dst_labels = dst_dir / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    n_images = 0
    n_person_bboxes = 0
    n_empty_after_filter = 0
    n_total_labels = 0
    n_orphan_labels = 0  # labels without a matching image

    for src_label in sorted(src_labels_dir.glob("*.txt")):
        n_total_labels += 1

        # Locate matching image first; skip if none found (orphan label)
        src_img = None
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = src_images_dir / (src_label.stem + ext)
            if candidate.exists():
                src_img = candidate
                break
        if src_img is None:
            n_orphan_labels += 1
            continue

        with open(src_label) as f:
            lines = f.readlines()

        person_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5 and int(parts[0]) == person_class_id:
                # Rewrite class id to 0 (single-class detector)
                person_lines.append(f"0 {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n")

        dst_label = dst_labels / src_label.name
        with open(dst_label, "w") as f:
            f.writelines(person_lines)

        link_or_copy(src_img, dst_images / src_img.name)
        n_images += 1

        n_person_bboxes += len(person_lines)
        if not person_lines:
            n_empty_after_filter += 1

    return {
        "label_files":   n_total_labels,
        "images":        n_images,
        "person_bboxes": n_person_bboxes,
        "empty_labels":  n_empty_after_filter,
        "orphan_labels": n_orphan_labels,
    }


print(f"PROCESSED_ROOT = {PROCESSED_ROOT}")


In [ ]:
# =============================================================================
# Chapter 7.3 - Run the filter on all four splits
# =============================================================================
# Splits, source directories, and the source dataset's "Person" class id.
SPLITS = [
    ("train",              CSS_DATA / "train" / "images",      CSS_DATA / "train" / "labels",      KAGGLE_PERSON_CLASS),
    ("val",                CSS_DATA / "valid" / "images",      CSS_DATA / "valid" / "labels",      KAGGLE_PERSON_CLASS),
    ("test_in_domain",     CSS_DATA / "test"  / "images",      CSS_DATA / "test"  / "labels",      KAGGLE_PERSON_CLASS),
    ("test_out_of_domain", ULTRALYTICS_DIR / "images" / "train", ULTRALYTICS_DIR / "labels" / "train", ULTRALYTICS_PERSON_CLASS),
]

# For Ultralytics out-of-domain test we want ALL of Ultralytics (train+val+test),
# since we never train on it. Extend the list with its other two folders.
EXTRA_OUT_OF_DOMAIN = [
    (ULTRALYTICS_DIR / "images" / "val",  ULTRALYTICS_DIR / "labels" / "val"),
    (ULTRALYTICS_DIR / "images" / "test", ULTRALYTICS_DIR / "labels" / "test"),
]

print(f"Writing processed dataset to: {PROCESSED_ROOT}")

# Start clean so re-runs do not leave stale files behind.
if PROCESSED_ROOT.exists():
    shutil.rmtree(PROCESSED_ROOT)
print(f"  (cleared previous contents)\n")

results = {}
for split_name, src_imgs, src_lbls, person_cls in SPLITS:
    dst_dir = PROCESSED_ROOT / split_name
    counts = filter_split(src_imgs, src_lbls, dst_dir, person_cls)
    results[split_name] = counts

# Extend test_out_of_domain with the rest of Ultralytics
for src_imgs, src_lbls in EXTRA_OUT_OF_DOMAIN:
    extra = filter_split(src_imgs, src_lbls,
                         PROCESSED_ROOT / "test_out_of_domain",
                         ULTRALYTICS_PERSON_CLASS)
    for k, v in extra.items():
        results["test_out_of_domain"][k] += v

# Report
print(f"{'Split':<22s} {'images':>8s} {'Person bboxes':>14s} {'empty labels':>14s} {'orphan labels':>14s}")
print("-" * 76)
for name, c in results.items():
    print(f"{name:<22s} {c['images']:>8,} {c['person_bboxes']:>14,} {c['empty_labels']:>14,} {c['orphan_labels']:>14,}")

print("\nNotes:")
print("  - 'empty labels' = images with no Person bbox (will serve as YOLO negatives).")
print("  - 'orphan labels' = label files without a matching image (skipped).")


In [ ]:
# =============================================================================
# Chapter 7.4 - Write data.yaml for YOLOv8 and verify the result
# =============================================================================
yaml_text = (
    f"# YOLOv8 dataset definition - Stage 1 (Person detection, single class)\n"
    f"# Generated by notebooks/01_main_pipeline.ipynb, chapter 7.\n"
    f"path: {PROCESSED_ROOT.as_posix()}\n"
    f"train: train/images\n"
    f"val: val/images\n"
    f"test: test_in_domain/images   # in-domain test (Kaggle CSS test)\n"
    f"\n"
    f"# Out-of-domain test set (entire Ultralytics dataset) is evaluated separately\n"
    f"# in chapter 9 by overriding the 'val' path to test_out_of_domain/images.\n"
    f"\n"
    f"nc: 1\n"
    f"names: ['Person']\n"
)
yaml_path = PROCESSED_ROOT / "data.yaml"
yaml_path.write_text(yaml_text)

print(f"Wrote: {yaml_path}\n")
print(yaml_text)

# Quick verification: image count vs label count must match in every split
print("Verification (images vs labels match):")
for split_name in ("train", "val", "test_in_domain", "test_out_of_domain"):
    split_dir = PROCESSED_ROOT / split_name
    n_imgs = sum(1 for _ in (split_dir / "images").glob("*"))
    n_lbls = sum(1 for _ in (split_dir / "labels").glob("*.txt"))
    status = "OK" if n_imgs == n_lbls else "MISMATCH"
    print(f"  {split_name:<22s} images={n_imgs:>5,}  labels={n_lbls:>5,}  [{status}]")


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">7.5 What was produced</h3>

<p style="text-align: justify;">
A processed dataset tree has been written to <code>DATA_ROOT/processed_stage1/</code>:
the four splits (<em>train</em>, <em>val</em>, <em>test_in_domain</em>, <em>test_out_of_domain</em>),
each containing an <code>images/</code> folder and a <code>labels/</code> folder, with Person as the single class
(remapped to id <code>0</code>). The accompanying <code>data.yaml</code> is what we will hand to YOLOv8 in Chapter&nbsp;8.
Images are hard-linked from the original datasets whenever possible, so no extra disk space is consumed.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">8. Stage 1 &mdash; YOLOv8 training for person detection</h2>

<p style="text-align: justify;">
This chapter fine-tunes <strong>YOLOv8n</strong> on the processed dataset from Chapter&nbsp;7 to detect people in
construction-site images. Training runs for 50 epochs on the <em>train</em> split, with the <em>val</em>
split used for early-stopping and best-checkpoint selection. The trained model is saved to
<code>results/stage1/yolov8n_50e/weights/best.pt</code> and will be used in
Chapter&nbsp;9 for evaluation and in Chapter&nbsp;10 to crop workers from images for Stage&nbsp;2.
</p>

<p style="text-align: justify; font-size: 13px; color: #475569;">
The training cell takes roughly 30&ndash;90 minutes on a workstation GPU (and many hours on CPU).
To allow re-running the notebook without re-training, the cell checks for existing weights at the target path
and loads them if present &mdash; controlled by the <code>SKIP_IF_TRAINED</code> flag.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 8.2 - Stage 1 training configuration
# =============================================================================
from ultralytics import YOLO

DATA_YAML       = PROCESSED_ROOT / "data.yaml"
STAGE1_DIR      = RESULTS_DIR / "stage1"
RUN_NAME        = "yolov8n_50e"
MODEL_VARIANT   = "yolov8n.pt"     # pretrained on COCO
EPOCHS          = 50
IMGSZ           = 640
BATCH           = -1               # -1 = auto (Ultralytics picks per GPU memory)
WORKERS         = 0                # 0 avoids Windows DataLoader multiprocessing issues

# Re-running the notebook should not re-train when weights already exist.
SKIP_IF_TRAINED = True
trained_weights = STAGE1_DIR / RUN_NAME / "weights" / "best.pt"

print("Stage 1 training plan")
print(f"  Model            : {MODEL_VARIANT}")
print(f"  Epochs           : {EPOCHS}")
print(f"  Image size       : {IMGSZ}")
print(f"  Batch            : {'auto' if BATCH == -1 else BATCH}")
print(f"  Data YAML        : {DATA_YAML}")
print(f"  Output directory : {STAGE1_DIR / RUN_NAME}")
print(f"  DataLoader workers: {WORKERS}")
print(f"  Skip if trained  : {SKIP_IF_TRAINED}")
print(f"  Existing weights : {'YES (will be loaded)' if trained_weights.exists() else 'no (will train)'}")


In [ ]:
# =============================================================================
# Chapter 8.3 - Train (or load existing weights)
# =============================================================================
if SKIP_IF_TRAINED and trained_weights.exists():
    print(f"Skipping training, loading existing weights:")
    print(f"  {trained_weights}")
    model = YOLO(str(trained_weights))
    train_results = None
else:
    print(f"Starting training - this may take 30-90 minutes on a GPU.")
    print()
    model = YOLO(MODEL_VARIANT)
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        project=str(STAGE1_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
    )
    print()
    print(f"Training complete. Best weights saved to:")
    print(f"  {trained_weights}")


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">8.4 Training complete</h3>

<p style="text-align: justify;">
The trained YOLO model is now available as <code>model</code> in memory. Its best weights are persisted to
<code>STAGE1_DIR / RUN_NAME / "weights" / "best.pt"</code> and will be used in Chapter&nbsp;9 for
quantitative evaluation (mAP, precision, recall) and in Chapter&nbsp;10 to crop workers from images for the
Stage&nbsp;2 multi-label classifier.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">9. Stage 1 evaluation</h2>

<p style="text-align: justify;">
With the YOLOv8n model trained in Chapter&nbsp;8, we now evaluate it on the two held-out test sets defined in Chapter&nbsp;4
&mdash; the first time these sets are touched in the notebook. The goal is to measure how well the model performs on its
own domain (Kaggle CSS test, in-domain) versus how well it generalises to a completely unseen domain (the Ultralytics
dataset, out-of-domain). The gap between the two numbers is the headline measure of the project narrative
<em>"the model succeeds in both worlds"</em>.
</p>

<p style="text-align: justify;">
We report the same metrics across all three splits (val, test in-domain, test out-of-domain) so they can be compared on
equal footing: mAP@0.5, mAP@0.5:0.95, precision, and recall.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 9.2 - Create test-specific data.yaml files
# =============================================================================
# Ultralytics .val() runs validation on the split named `val` inside its
# data.yaml. To evaluate on each test split we create two tiny YAML files
# that point `val` at the desired test directory.
import yaml

EVAL_DIR = STAGE1_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

IN_DOMAIN_YAML = EVAL_DIR / "data_test_in_domain.yaml"
OOD_YAML       = EVAL_DIR / "data_test_out_of_domain.yaml"

base_yaml = {
    "path":  PROCESSED_ROOT.as_posix(),
    "train": "train/images",      # required field; not used during .val()
    "nc":    1,
    "names": ["Person"],
}

# In-domain: Kaggle CSS test (80 images)
with open(IN_DOMAIN_YAML, "w") as f:
    yaml.safe_dump({**base_yaml, "val": "test_in_domain/images"}, f, sort_keys=False)

# Out-of-domain: Ultralytics, entire dataset (1,414 images)
with open(OOD_YAML, "w") as f:
    yaml.safe_dump({**base_yaml, "val": "test_out_of_domain/images"}, f, sort_keys=False)

print(f"Wrote {IN_DOMAIN_YAML}")
print(f"Wrote {OOD_YAML}")


In [ ]:
# =============================================================================
# Chapter 9.3 - Run model.val() on each test set
# =============================================================================
# The trained `model` object from Chapter 8 is still in memory (best.pt loaded).
# Each call to .val() produces a full evaluation: PR curve, confusion matrix,
# and a DetMetrics object holding the numeric results.

print("=" * 70)
print("Test In-Domain - Kaggle CSS test (80 images, 172 Person bboxes)")
print("=" * 70)
metrics_in_domain = model.val(
    data=str(IN_DOMAIN_YAML),
    project=str(STAGE1_DIR / "evaluation"),
    name="test_in_domain",
    exist_ok=True,
    workers=0,
    verbose=False,
)

print()
print("=" * 70)
print("Test Out-of-Domain - Ultralytics ALL (1,414 images, 2,244 Person bboxes)")
print("=" * 70)
metrics_ood = model.val(
    data=str(OOD_YAML),
    project=str(STAGE1_DIR / "evaluation"),
    name="test_out_of_domain",
    exist_ok=True,
    workers=0,
    verbose=False,
)


In [ ]:
# =============================================================================
# Chapter 9.4 - Compile metrics and compute Cross-Dataset Gap
# =============================================================================
# Pull metrics out of the DetMetrics objects.
# The validation numbers from Chapter 8 are hard-coded here for reference
# (they came from the best.pt selection on Kaggle val).

results = pd.DataFrame([
    {
        "Split":        "Validation (Kaggle val)",
        "Images":       114,
        "Person bboxes": 166,
        "mAP@0.5":      0.829,
        "mAP@0.5:0.95": 0.509,
        "Precision":    0.85,
        "Recall":       0.753,
    },
    {
        "Split":        "Test In-Domain (Kaggle test)",
        "Images":       80,
        "Person bboxes": 172,
        "mAP@0.5":      float(metrics_in_domain.box.map50),
        "mAP@0.5:0.95": float(metrics_in_domain.box.map),
        "Precision":    float(metrics_in_domain.box.mp),
        "Recall":       float(metrics_in_domain.box.mr),
    },
    {
        "Split":        "Test Out-of-Domain (Ultralytics)",
        "Images":       1414,
        "Person bboxes": 2244,
        "mAP@0.5":      float(metrics_ood.box.map50),
        "mAP@0.5:0.95": float(metrics_ood.box.map),
        "Precision":    float(metrics_ood.box.mp),
        "Recall":       float(metrics_ood.box.mr),
    },
])

# Pretty-print: per-column formatters so the string "Split" column is not
# touched (it would clash with the int/float number-format specifiers).
formatters = {
    "Images":        lambda v: f"{v:,}",
    "Person bboxes": lambda v: f"{v:,}",
    "mAP@0.5":       lambda v: f"{v:.3f}",
    "mAP@0.5:0.95":  lambda v: f"{v:.3f}",
    "Precision":     lambda v: f"{v:.3f}",
    "Recall":        lambda v: f"{v:.3f}",
}
print(results.to_string(index=False, formatters=formatters))

# Cross-Dataset Gap (In-Domain minus Out-of-Domain)
gap_map50  = results.loc[1, "mAP@0.5"]      - results.loc[2, "mAP@0.5"]
gap_map    = results.loc[1, "mAP@0.5:0.95"] - results.loc[2, "mAP@0.5:0.95"]
gap_p      = results.loc[1, "Precision"]    - results.loc[2, "Precision"]
gap_r      = results.loc[1, "Recall"]       - results.loc[2, "Recall"]

print()
print("=" * 70)
print("Cross-Dataset Gap  =  In-Domain  -  Out-of-Domain")
print("=" * 70)
print(f"  mAP@0.5      : {gap_map50:+.3f}  (positive = in-domain better)")
print(f"  mAP@0.5:0.95 : {gap_map:+.3f}")
print(f"  Precision    : {gap_p:+.3f}")
print(f"  Recall       : {gap_r:+.3f}")

# Save for downstream reporting
results.to_csv(STAGE1_DIR / "evaluation" / "summary.csv", index=False)
print(f"\nSaved summary to: {STAGE1_DIR / 'evaluation' / 'summary.csv'}")


In [ ]:
# =============================================================================
# Chapter 9.5 - Visualise the comparison
# =============================================================================
metrics_to_plot = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]
short_labels   = ["Val\n(Kaggle val)", "Test In-Domain\n(Kaggle test)", "Test Out-of-Domain\n(Ultralytics)"]
colors         = ["#94a3b8", "#1e40af", "#dc2626"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, metric in zip(axes, metrics_to_plot):
    values = results[metric].values
    bars = ax.bar(short_labels, values, color=colors, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(metric, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Stage 1 - Person detection - across val and both test sets", fontsize=13)
plt.tight_layout()
plt.savefig(STAGE1_DIR / "evaluation" / "comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {STAGE1_DIR / 'evaluation' / 'comparison.png'}")


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">9.6 Stage 1 evaluation summary</h3>

<p style="text-align: justify;">
The trained YOLOv8n model was evaluated on three splits. The numbers below summarise the headline findings.
</p>

<table style="border-collapse: collapse; font-size: 13.5px; margin-top: 6px;">
  <thead><tr>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Split</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">mAP@0.5</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">mAP@0.5:0.95</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Precision</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Recall</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Val (Kaggle val)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.829</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.509</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.850</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.753</td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Test In-Domain (Kaggle test)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.816</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.507</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.808</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.756</td>
    </tr>
    <tr style="background:#fef3c7;">
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>Test Out-of-Domain (Ultralytics)</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>0.617</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>0.213</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>0.693</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>0.604</strong></td>
    </tr>
  </tbody>
</table>

<h4 style="color: #1e40af; margin-top: 18px;">Key findings</h4>

<ol>
  <li style="margin-bottom: 8px;"><strong>No overfitting to validation.</strong> The model performs almost identically on val and Kaggle test (mAP@0.5 of 0.829 vs 0.816) &mdash; confirming that <code>best.pt</code> represents the true model capacity on the trained domain.</li>
  <li style="margin-bottom: 8px;"><strong>Substantial Cross-Dataset Gap.</strong> Performance drops by ~20&nbsp;pp on mAP@0.5 and ~30&nbsp;pp on mAP@0.5:0.95 when moving to the out-of-domain (Ultralytics) split. The gap is most extreme on tight-box accuracy &mdash; the model finds approximate locations of people, but struggles with precise box placement on the unseen domain.</li>
  <li style="margin-bottom: 8px;"><strong>Recall degradation matters most.</strong> Recall drops from 0.75 to 0.60 &mdash; the model misses 40% of workers in Ultralytics. For PPE compliance, this is the operational bottleneck: a missed worker is a missed potential non-compliance event.</li>
</ol>

<h4 style="color: #1e40af; margin-top: 18px;">Implications for the project narrative</h4>

<p style="text-align: justify;">
The original framing of <em>"the model succeeds in both worlds"</em> requires refinement based on these numbers. The model is stable and successful on the trained domain (Kaggle), and partially generalises to a new domain (Ultralytics) but with a noticeable gap. The 20&nbsp;pp Cross-Dataset Gap is a real and academically interesting finding, and will be the central subject of the discussion in Chapter&nbsp;15.
</p>

<p style="text-align: justify;">
Stage 2 (PPE classification on cropped persons, Chapter&nbsp;11) will work with reliable crops on the In-Domain images and noisier crops on the Out-of-Domain images. End-to-end evaluation in Chapter&nbsp;14 will tell us whether the classifier can partially compensate for Stage 1's missed detections.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">10. Build crop dataset for Stage 2</h2>

<p style="text-align: justify;">
Stage 2 is a per-worker classifier &mdash; it takes a single cropped image of one person and returns
two independent binary labels (helmet, vest). To train it, we need a dataset of <em>person crops</em>
where each crop already has its helmet/vest ground truth attached.
</p>

<p style="text-align: justify;">
This chapter takes the original (multi-class) labels of Kaggle CSS and Ultralytics, iterates over every
ground-truth Person bounding box, crops the corresponding image patch, and assigns labels using a simple rule:
a Person crop is marked <code>helmet=1</code> if any Hardhat bbox in the same image overlaps with this person's
bbox (IoU&nbsp;&gt;&nbsp;0.1 or the item's center sits inside the person's bbox), and analogously for <code>vest</code>.
If no PPE item is associated with the person, the labels default to 0
(Strategy&nbsp;A from the discussion in Chapter&nbsp;9).
</p>

<p style="text-align: justify;">
We build the crop dataset for all four splits at once: <em>train</em> and <em>val</em> (for Stage 2 training),
plus <em>test_in_domain</em> (Kaggle test) and <em>test_out_of_domain</em> (the entire Ultralytics dataset)
for Stage 2 evaluation in Chapter&nbsp;12.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 10.2 - Configuration and helper functions
# =============================================================================
import shutil
import cv2

# Output root for Stage 2 crops
STAGE2_CROPS_ROOT = DATA_ROOT / "stage2_crops"

# --- Class IDs ---------------------------------------------------------------
# Kaggle CSS (10 classes)
KAGGLE_PERSON_CLS      = 5
KAGGLE_HARDHAT_CLS     = 0
KAGGLE_VEST_CLS        = 7    # "Safety Vest"

# Ultralytics Construction-PPE (11 classes)
ULT_PERSON_CLS         = 6
ULT_HELMET_CLS         = 0    # "helmet"
ULT_VEST_CLS           = 2    # "vest"

# --- Labeling rule -----------------------------------------------------------
# A PPE item is considered "worn by" a person if either:
#   (a) IoU(person_bbox, item_bbox) > IOU_THRESHOLD, OR
#   (b) The item's center sits inside the person bbox.
IOU_THRESHOLD = 0.10
MIN_CROP_SIDE = 16    # skip crops smaller than this (in pixels) - too small to learn from


# --- Geometry helpers --------------------------------------------------------
def yolo_to_pixel(cx, cy, w, h, img_w, img_h):
    """Convert YOLO normalized (cx, cy, w, h) -> pixel (x1, y1, x2, y2)."""
    xc, yc = cx * img_w, cy * img_h
    bw, bh = w * img_w, h * img_h
    x1 = max(0, int(round(xc - bw / 2)))
    y1 = max(0, int(round(yc - bh / 2)))
    x2 = min(img_w, int(round(xc + bw / 2)))
    y2 = min(img_h, int(round(yc + bh / 2)))
    return x1, y1, x2, y2


def compute_iou(a, b):
    """IoU between two (x1, y1, x2, y2) boxes."""
    xa, ya = max(a[0], b[0]), max(a[1], b[1])
    xb, yb = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def is_worn_by(person_box, item_box, iou_thr=IOU_THRESHOLD):
    """True if item_box belongs to the person (overlap or center inside)."""
    if compute_iou(person_box, item_box) > iou_thr:
        return True
    cx = (item_box[0] + item_box[2]) / 2
    cy = (item_box[1] + item_box[3]) / 2
    return (person_box[0] <= cx <= person_box[2] and
            person_box[1] <= cy <= person_box[3])


In [ ]:
# =============================================================================
# Chapter 10.3 - Build crops for all 4 splits
# =============================================================================
def find_image(images_dir, stem):
    for ext in (".jpg", ".jpeg", ".png"):
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    return None


def process_one_label_file(label_file, images_dir, person_cls, helmet_cls, vest_cls):
    """Parse one .txt file, return (image_pixel_array, [(person_box, helmet, vest), ...])"""
    img_file = find_image(images_dir, label_file.stem)
    if img_file is None:
        return None, []

    img = cv2.imread(str(img_file))
    if img is None:
        return None, []
    H, W = img.shape[:2]

    person_boxes, helmet_boxes, vest_boxes = [], [], []
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid = int(parts[0])
            box = yolo_to_pixel(*map(float, parts[1:]), W, H)
            if cid == person_cls:    person_boxes.append(box)
            elif cid == helmet_cls:  helmet_boxes.append(box)
            elif cid == vest_cls:    vest_boxes.append(box)

    results = []
    for p_box in person_boxes:
        x1, y1, x2, y2 = p_box
        if (x2 - x1) < MIN_CROP_SIDE or (y2 - y1) < MIN_CROP_SIDE:
            continue
        helmet = int(any(is_worn_by(p_box, h) for h in helmet_boxes))
        vest   = int(any(is_worn_by(p_box, v) for v in vest_boxes))
        results.append((p_box, helmet, vest))
    return img, results


def build_split_crops(sources, dst_dir, split_tag, person_cls, helmet_cls, vest_cls):
    """`sources` = list of (images_dir, labels_dir) tuples (Ultralytics has 3, Kaggle has 1)."""
    crops_dir = dst_dir / "crops"
    crops_dir.mkdir(parents=True, exist_ok=True)

    records = []
    n_skipped_tiny = 0

    for images_dir, labels_dir in sources:
        for label_file in sorted(labels_dir.glob("*.txt")):
            img, persons = process_one_label_file(
                label_file, images_dir, person_cls, helmet_cls, vest_cls
            )
            if img is None:
                continue
            for p_idx, (p_box, helmet, vest) in enumerate(persons):
                x1, y1, x2, y2 = p_box
                crop = img[y1:y2, x1:x2]
                if crop.size == 0 or crop.shape[0] < MIN_CROP_SIDE or crop.shape[1] < MIN_CROP_SIDE:
                    n_skipped_tiny += 1
                    continue
                crop_name = f"{split_tag}__{label_file.stem}__p{p_idx}.jpg"
                cv2.imwrite(str(crops_dir / crop_name), crop)
                records.append({
                    "filename":     crop_name,
                    "source_image": label_file.stem,
                    "person_idx":   p_idx,
                    "helmet":       helmet,
                    "vest":         vest,
                })

    df = pd.DataFrame(records)
    df.to_csv(dst_dir / "labels.csv", index=False)
    return df, n_skipped_tiny


# --- Clear previous run ------------------------------------------------------
if STAGE2_CROPS_ROOT.exists():
    shutil.rmtree(STAGE2_CROPS_ROOT)
print(f"Writing crops to: {STAGE2_CROPS_ROOT}\n")

# --- Build each split --------------------------------------------------------
splits_config = [
    # (output_subdir, sources, split_tag, person_cls, helmet_cls, vest_cls)
    ("train",
     [(CSS_DATA / "train" / "images",  CSS_DATA / "train" / "labels")],
     "train", KAGGLE_PERSON_CLS, KAGGLE_HARDHAT_CLS, KAGGLE_VEST_CLS),

    ("val",
     [(CSS_DATA / "valid" / "images",  CSS_DATA / "valid" / "labels")],
     "val", KAGGLE_PERSON_CLS, KAGGLE_HARDHAT_CLS, KAGGLE_VEST_CLS),

    ("test_in_domain",
     [(CSS_DATA / "test"  / "images",  CSS_DATA / "test"  / "labels")],
     "indom_test", KAGGLE_PERSON_CLS, KAGGLE_HARDHAT_CLS, KAGGLE_VEST_CLS),

    ("test_out_of_domain",
     [(ULTRALYTICS_DIR / "images" / s, ULTRALYTICS_DIR / "labels" / s)
      for s in ("train", "val", "test")],
     "ood", ULT_PERSON_CLS, ULT_HELMET_CLS, ULT_VEST_CLS),
]

summary = []
for subdir, sources, tag, p_cls, h_cls, v_cls in splits_config:
    df, n_tiny = build_split_crops(
        sources, STAGE2_CROPS_ROOT / subdir, tag, p_cls, h_cls, v_cls,
    )
    n_total = len(df)
    n_helmet = int(df["helmet"].sum())
    n_vest = int(df["vest"].sum())
    n_compliant = int(((df["helmet"] == 1) & (df["vest"] == 1)).sum())
    summary.append({
        "Split":             subdir,
        "Crops":             n_total,
        "Helmet (=1)":       n_helmet,
        "Helmet %":          100 * n_helmet / max(1, n_total),
        "Vest (=1)":         n_vest,
        "Vest %":            100 * n_vest / max(1, n_total),
        "Compliant (h&v)":   n_compliant,
        "Compliant %":       100 * n_compliant / max(1, n_total),
        "Skipped (<16px)":   n_tiny,
    })

stats = pd.DataFrame(summary)
print(stats.to_string(
    index=False,
    formatters={
        "Crops":           lambda v: f"{v:,}",
        "Helmet (=1)":     lambda v: f"{v:,}",
        "Vest (=1)":       lambda v: f"{v:,}",
        "Compliant (h&v)": lambda v: f"{v:,}",
        "Skipped (<16px)": lambda v: f"{v:,}",
        "Helmet %":        lambda v: f"{v:5.1f}",
        "Vest %":          lambda v: f"{v:5.1f}",
        "Compliant %":     lambda v: f"{v:5.1f}",
    },
))


In [ ]:
# =============================================================================
# Chapter 10.4 - Sample a few crops from each split for sanity check
# =============================================================================
# Load each split's labels.csv and show 2 random crops per split, with their labels.

random.seed(SEED)

def crop_caption(row):
    flags = []
    if row["helmet"]: flags.append("H")
    if row["vest"]:   flags.append("V")
    return f"H={row['helmet']} V={row['vest']}"


fig, axes = plt.subplots(4, 3, figsize=(12, 14))
for row_idx, subdir in enumerate(["train", "val", "test_in_domain", "test_out_of_domain"]):
    csv_path = STAGE2_CROPS_ROOT / subdir / "labels.csv"
    df = pd.read_csv(csv_path)
    if len(df) < 3:
        for col in range(3):
            axes[row_idx, col].text(0.5, 0.5, "no crops", ha="center")
            axes[row_idx, col].axis("off")
        continue
    samples = df.sample(n=3, random_state=SEED + row_idx).reset_index(drop=True)
    for col in range(3):
        ax = axes[row_idx, col]
        row = samples.iloc[col]
        img_path = STAGE2_CROPS_ROOT / subdir / "crops" / row["filename"]
        if img_path.exists():
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        ax.set_title(f"{subdir}\n{crop_caption(row)}", fontsize=9)
        ax.axis("off")

plt.suptitle("Stage 2 crop samples - one row per split, 3 random crops each",
             fontsize=13, y=0.995)
plt.tight_layout()
plt.savefig(STAGE2_CROPS_ROOT / "samples.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved samples preview to: {STAGE2_CROPS_ROOT / 'samples.png'}")


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">10.5 Stage 2 crop dataset &mdash; summary</h3>

<p style="text-align: justify;">
The crop dataset under <code>DATA_ROOT/stage2_crops/</code> now contains four splits (<em>train</em>, <em>val</em>,
<em>test_in_domain</em>, <em>test_out_of_domain</em>), each with a <code>crops/</code> folder of JPEG images and
a <code>labels.csv</code> file mapping every crop to its (helmet, vest) labels.
Chapter&nbsp;11 will train a CNN on the train+val splits; Chapter&nbsp;12 will evaluate it on the two test sets.
</p>

<div style="background: #fef3c7; border: 1px solid #fbbf24; padding: 8px 12px; border-radius: 4px; color: #78350f; font-size: 13px;">
<em><strong>Note on Strategy A labels.</strong> A crop receives <code>helmet=0</code> whenever no Hardhat bbox is found
near the person. This includes both genuine no-helmet cases and people simply not annotated for PPE. The classifier
in Chapter&nbsp;11 may therefore learn a slightly noisy decision boundary; this trade-off was deliberately accepted
in exchange for a larger, simpler training set.</em>
</div>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">11. Stage 2 &mdash; CNN multi-label classifier (helmet, vest)</h2>

<p style="text-align: justify;">
Stage 2 takes a single person crop produced by Chapter&nbsp;10 and predicts two independent labels &mdash;
<code>helmet</code> and <code>vest</code> &mdash; via a CNN built on a pretrained ImageNet backbone.
Architecturally this mirrors Unit&nbsp;8 of the course (Cats vs Dogs): a pretrained backbone topped with
a small Dense+Sigmoid classifier head, multi-label BCE loss.
</p>

<div style="background:#fef3c7; border:1px solid #fbbf24; padding:10px 14px;
            border-radius:4px; color:#78350f;">
<strong>&#9881; Why training happens in a separate notebook.</strong>
Stage 2 is built with Keras / TensorFlow. From TF&nbsp;2.11 onwards, Google dropped GPU support on
native Windows &mdash; only WSL2 or DirectML plugins are supported. On the project workstation that
means TF would fall back to CPU and a single epoch would take many minutes. The Colab notebook
<code>stage2_colab.ipynb</code> sidesteps that with a free T4 / L4 GPU, completing the full Stage 2
training in roughly 20&nbsp;minutes total. This main notebook only loads and displays the artifacts
produced there.
</div>

<h3 style="color: #1e40af; margin-top: 18px;">11.1 Three-stage methodology</h3>

<p>Following the structure of Unit&nbsp;8, Stage 2 proceeds in three sub-stages:</p>

<table style="border-collapse: collapse; font-size: 13.5px; margin-top: 6px;">
  <thead><tr>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Sub-stage</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">What we do</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Where</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>2.1 Ablation</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Compare two architectures (VGG16, ResNet50) under identical conditions &mdash; both with a frozen backbone (feature extraction), 20 epochs.</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><code>stage2_colab.ipynb</code></td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>2.2 Refinement</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Take the winner of 2.1, unfreeze its top 30 backbone layers, continue training 15 epochs with a 10&times; lower learning rate (fine-tuning).</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><code>stage2_colab.ipynb</code></td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>2.3 Evaluation</strong></td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Final model evaluated once on the two held-out test sets (Kaggle test, Ultralytics).</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Chapter 12</td>
    </tr>
  </tbody>
</table>

<p style="font-size: 13px; color: #475569; margin-top: 10px;">
The artifacts loaded in the cells below all come from <code>stage2_colab.ipynb</code> and live under
<code>results/stage2/</code>.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 11.2 - Summary table across all three sub-stages (best val AUC each)
# =============================================================================
STAGE2_DIR = REPO_ROOT / "results" / "stage2"

models = {
    "VGG16     (2.1 ablation, frozen backbone)":     STAGE2_DIR / "vgg16"              / "history.csv",
    "ResNet50  (2.1 ablation, frozen backbone)":     STAGE2_DIR / "resnet50"           / "history.csv",
    "ResNet50  (2.2 refinement, fine-tuned)":        STAGE2_DIR / "resnet50_finetuned" / "history.csv",
}

rows = []
for name, csv_path in models.items():
    h = pd.read_csv(csv_path)
    best_idx = int(h["val_auc"].idxmax())
    rows.append({
        "Model":     name,
        "Epochs":    len(h),
        "loss":      h.loc[best_idx, "val_loss"],
        "accuracy":  h.loc[best_idx, "val_accuracy"],
        "AUC":       h.loc[best_idx, "val_auc"],
        "precision": h.loc[best_idx, "val_precision"],
        "recall":    h.loc[best_idx, "val_recall"],
    })

training_summary = pd.DataFrame(rows)
formatters = {
    "Epochs":    lambda v: f"{v:,}",
    "loss":      lambda v: f"{v:.4f}",
    "accuracy":  lambda v: f"{v:.4f}",
    "AUC":       lambda v: f"{v:.4f}",
    "precision": lambda v: f"{v:.4f}",
    "recall":    lambda v: f"{v:.4f}",
}
print("Best-AUC validation metrics across the three sub-stages")
print("=" * 100)
print(training_summary.to_string(index=False, formatters=formatters))


In [ ]:
# =============================================================================
# Chapter 11.3 - Ablation training curves: VGG16 vs ResNet50 (feature extraction)
# =============================================================================
import matplotlib.image as mpimg
ablation_img = mpimg.imread(STAGE2_DIR / "comparison.png")
fig, ax = plt.subplots(figsize=(15, 5))
ax.imshow(ablation_img)
ax.axis("off")
ax.set_title("Stage 2.1 - Ablation: training curves of VGG16 and ResNet50 (frozen backbones)",
             fontsize=11)
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">11.4 Reading the ablation result</h3>

<p style="text-align: justify;">
ResNet50 wins the ablation cleanly &mdash; <strong>+5&nbsp;pp val AUC</strong> (0.949 vs 0.900) and
<strong>+14&nbsp;pp val recall</strong> (0.829 vs 0.694) over VGG16. The dramatic recall gap matters
for PPE compliance: VGG16's high precision but lower recall means it skips many actually-compliant
workers, while ResNet50 finds them. The two characteristics that probably drive the gap are
ResNet50's <strong>residual connections</strong> (cleaner gradient flow during the head training)
and its <strong>smaller parameter count</strong> (25M vs 138M), which is easier to regularise on a
modest dataset.
</p>

<p style="text-align: justify;">
We discard VGG16 from now on but keep its checkpoint and history as a baseline for the discussion
in Chapter&nbsp;15. The next sub-stage continues with ResNet50 only.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 11.5 - Fine-tuning trajectory: ResNet50 FE -> FT
# =============================================================================
ft_img = mpimg.imread(STAGE2_DIR / "finetune_trajectory.png")
fig, ax = plt.subplots(figsize=(15, 5))
ax.imshow(ft_img)
ax.axis("off")
ax.set_title("Stage 2.2 - Refinement: ResNet50 feature extraction -> fine-tuning (epoch 21 onwards)",
             fontsize=11)
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">11.6 Reading the fine-tuning result</h3>

<p style="text-align: justify;">
After 20 frozen-backbone epochs, the top 30 layers of the ResNet50 backbone are unfrozen and training
continues for up to 15 more epochs at <code>lr&nbsp;=&nbsp;1e-5</code> (10&times; lower than the
feature-extraction phase). The vertical line in the plots marks where fine-tuning begins (epoch&nbsp;21).
</p>

<p style="text-align: justify;">
Fine-tuning delivers two clear improvements over the feature-extraction-only ResNet50:
</p>

<ul>
  <li><strong>val AUC: 0.949 &rarr; 0.974</strong> (+2.5&nbsp;pp). Smaller than Unit&nbsp;8's
      14&nbsp;pp Cats-vs-Dogs jump &mdash; expected, since the val set is only 150 crops and the
      backbone was already strong.</li>
  <li><strong>val precision: 0.836 &rarr; 0.939</strong> (+10&nbsp;pp). The headline improvement:
      the model becomes substantially more confident, with far fewer false positives, while
      <em>recall stays at the same level</em> (0.829 &rarr; 0.838).</li>
</ul>

<p style="text-align: justify;">
The train loss curve drops steeply once fine-tuning starts (from ~0.20 down to ~0.07), while val
loss remains flat around 0.20-0.24. That gap is mild overfitting, but it is tracked correctly by
<code>restore_best_weights=True</code>: the saved <code>best.keras</code> is the highest val-AUC
epoch, not the last one.
</p>

<p style="text-align: justify;">
<strong>The fine-tuned ResNet50 is now the final Stage 2 model.</strong> Chapter&nbsp;12 evaluates
it once on the two held-out test sets.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">12. Stage 2 evaluation</h2>

<p style="text-align: justify;">
We now evaluate the final Stage 2 model &mdash; the fine-tuned ResNet50 from Chapter&nbsp;11 &mdash;
on the two held-out test sets we set aside in Chapter&nbsp;4:
</p>

<ul>
  <li><strong>Test In-Domain</strong>: Kaggle CSS test crops (155 person crops). Same domain the model was trained on.</li>
  <li><strong>Test Out-of-Domain</strong>: Ultralytics dataset crops (2,243 person crops). A domain the model has never seen during training.</li>
</ul>

<p style="text-align: justify;">
The gap between the two numbers is the <strong>Stage 2 Cross-Dataset Gap</strong> &mdash; the empirical
answer to <em>"does the classifier generalize beyond the training domain?"</em>.
Comparing it to the Stage&nbsp;1 Cross-Dataset Gap from Chapter&nbsp;9 also tells us
<em>which stage in the pipeline is more fragile under domain shift</em>.
</p>

<p style="text-align: justify;">
The evaluation itself ran inside <code>stage2_colab.ipynb</code> (Colab GPU). The cells below
just load the resulting <code>summary.csv</code> and the rendered bar chart.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 12.2 - Load the Stage 2 evaluation summary
# =============================================================================
STAGE2_EVAL_DIR = REPO_ROOT / "results" / "stage2" / "evaluation"

eval_summary = pd.read_csv(STAGE2_EVAL_DIR / "summary.csv")

formatters = {
    "Crops":     lambda v: f"{int(v):,}",
    "loss":      lambda v: f"{v:.4f}",
    "accuracy":  lambda v: f"{v:.4f}",
    "AUC":       lambda v: f"{v:.4f}",
    "precision": lambda v: f"{v:.4f}",
    "recall":    lambda v: f"{v:.4f}",
}
print("Stage 2 final evaluation - ResNet50 fine-tuned, across val and both test sets")
print("=" * 100)
print(eval_summary.to_string(index=False, formatters=formatters))


In [ ]:
# =============================================================================
# Chapter 12.3 - Cross-Dataset Gap (In-Domain - Out-of-Domain)
# =============================================================================
in_dom = eval_summary.loc[1]   # row 1: Test In-Domain
ood    = eval_summary.loc[2]   # row 2: Test Out-of-Domain

gap = {
    "AUC":       in_dom["AUC"]       - ood["AUC"],
    "precision": in_dom["precision"] - ood["precision"],
    "recall":    in_dom["recall"]    - ood["recall"],
    "accuracy":  in_dom["accuracy"]  - ood["accuracy"],
}

print("Cross-Dataset Gap  =  In-Domain  -  Out-of-Domain")
print("=" * 60)
for m, v in gap.items():
    sign = "+" if v >= 0 else ""
    print(f"  {m:<12s}: {sign}{v:.4f}  ({sign}{v*100:.1f} pp)")

print()
print("Reference for comparison - Stage 1 Cross-Dataset Gap (Chapter 9):")
print("  mAP@0.5     : +0.199  (+19.9 pp)")
print("  mAP@0.5:0.95: +0.294  (+29.4 pp)")
print("  Recall      : +0.149  (+14.9 pp)")
print("  Precision   : +0.114  (+11.4 pp)")


In [ ]:
# =============================================================================
# Chapter 12.4 - Bar chart comparison across val and both test sets
# =============================================================================
import matplotlib.image as mpimg
eval_img = mpimg.imread(STAGE2_EVAL_DIR / "comparison.png")
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.imshow(eval_img)
ax.axis("off")
ax.set_title("Stage 2.3 - ResNet50 fine-tuned: val vs in-domain vs out-of-domain",
             fontsize=11)
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">12.5 Findings</h3>

<h4 style="color:#1e40af;">12.5.1 Test In-Domain confirms the val number</h4>
<p style="text-align: justify;">
Test In-Domain marginally <em>outperforms</em> val (AUC 0.979 vs 0.974, recall 0.906 vs 0.838). With
only 150 val crops, that small swap is well within statistical noise. The takeaway is that
<code>best.keras</code> truly captures the model's behaviour on Kaggle &mdash; not an overfit to the
specific 150-crop val set.
</p>

<h4 style="color:#1e40af;">12.5.2 The Cross-Dataset Gap is real but moderate</h4>
<p style="text-align: justify;">
On Ultralytics (Out-of-Domain), AUC drops by ~7&nbsp;pp (0.979 &rarr; 0.912) and recall by ~17&nbsp;pp
(0.906 &rarr; 0.731). Importantly, <strong>precision barely moves</strong> (0.954 &rarr; 0.938).
The model stays cautious on the unseen domain: when it does say "this person is wearing a helmet",
it is right 94% of the time even on data it has never seen. What it loses is sensitivity &mdash;
it skips more of the actually-compliant workers.
</p>

<p style="text-align: justify;">
The dominant driver of the recall drop is the <strong>class-prior shift</strong>: in the training
data only ~33% of crops are positive (helmet&nbsp;=&nbsp;1), while in Ultralytics ~69% are. The model
learned <em>"default = no helmet"</em>, and on a domain where the default is reversed it underpredicts
the positive class. This is a known failure mode of multi-label classifiers trained under
class-prior mismatch &mdash; not a fault of the architecture.
</p>

<h4 style="color:#1e40af;">12.5.3 Stage 2 generalizes much better than Stage 1</h4>

<table style="border-collapse: collapse; font-size: 13.5px; margin-top: 6px;">
  <thead><tr>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Pipeline stage</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Headline metric</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">In-Domain</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Out-of-Domain</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Gap</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Stage 1 (YOLOv8n)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">mAP@0.5</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.816</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.617</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>+0.199</strong> (~20 pp)</td>
    </tr>
    <tr style="background:#f0fdf4;">
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Stage 2 (ResNet50 fine-tuned)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">AUC</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.979</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.912</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>+0.067</strong> (~7 pp)</td>
    </tr>
  </tbody>
</table>

<p style="text-align: justify; margin-top: 10px;">
The Stage 2 classifier loses <strong>about one third</strong> of what Stage 1 loses. This validates
the two-stage architecture choice. The classifier sees only a focused crop of a person with no
shifting background, and the residual ImageNet features of "helmet colour / vest texture" transfer
well from Kaggle to Ultralytics. The detector, in contrast, has to localise people against an
unseen visual scene &mdash; a much harder generalisation problem.
</p>

<h4 style="color:#1e40af;">12.5.4 Updated project narrative</h4>
<p style="text-align: justify;">
The original framing of <em>"the model succeeds in both worlds"</em> can now be supported with
numbers: on the trained domain it is excellent (AUC 0.98), on a completely unseen domain it remains
strong (AUC 0.91, precision 0.94). The classifier's biggest remaining weakness is recall on the
unseen domain (0.73), driven mainly by class-prior shift &mdash; an avenue for future work,
discussed in Chapter&nbsp;15.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">13. Stage 3 &mdash; Compliance Logic</h2>

<p style="text-align: justify;">
The first two stages are the <em>perception</em> half of the pipeline: Stage&nbsp;1 (YOLOv8n)
answers <em>"where are the people?"</em> and Stage&nbsp;2 (ResNet50) answers, for each cropped
person, <em>"is a helmet present?"</em> and <em>"is a vest present?"</em>. Stage&nbsp;3 is the
<em>policy</em> half: it turns those two boolean flags into a single, auditable
<strong>compliance verdict</strong> for the worker.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">13.1 The rule</h3>
<p style="text-align: justify;">
The site policy in this project requires <strong>both</strong> a helmet and a high-visibility
vest. A worker is therefore compliant <em>if and only if</em> both items are present:
</p>

<p style="text-align: center; font-size: 15px;">
<code>compliant = helmet AND vest</code>
</p>

<p style="text-align: justify;">
When the worker is not compliant, the verdict also lists <strong>which</strong> items are missing,
so the output is <em>actionable</em> &mdash; a supervisor can see at a glance whether a worker needs
a helmet, a vest, or both.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">13.2 Why a separate, model-free stage</h3>
<ul>
  <li><strong>Separation of perception from policy.</strong> The networks learn what is <em>in</em> the
      image; the rule encodes what the site <em>requires</em>. Keeping them apart means the policy can
      change &mdash; e.g. a site that also mandates gloves &mdash; without retraining a single weight.</li>
  <li><strong>Transparency / auditability.</strong> The verdict is a one-line boolean expression, not a
      black box. Every "non-compliant" decision can be traced to a specific missing item.</li>
  <li><strong>Clean error attribution.</strong> Because Stage&nbsp;3 is deterministic, any mistake in the
      final verdict is fully attributable to a Stage&nbsp;1 or Stage&nbsp;2 perception error &mdash; never
      to the policy logic itself.</li>
</ul>

<p style="text-align: justify;">
<strong>Threshold boundary.</strong> Stage&nbsp;2 emits a <em>probability</em> per PPE item. We convert
those probabilities to the boolean flags this stage consumes by thresholding at <code>0.5</code> at the
Stage&nbsp;2&nbsp;&rarr;&nbsp;Stage&nbsp;3 boundary. That conversion is applied in the end-to-end pipeline
of Chapter&nbsp;14; here we work directly with the resulting booleans.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 13.3 - Stage 3: the compliance() policy function
# =============================================================================
from typing import Dict, List

# Site policy: which PPE items are mandatory. Listing them here (rather than
# hard-coding "helmet and vest") is what lets the policy change without
# touching either trained model - e.g. adding "gloves" later.
REQUIRED_PPE = ("helmet", "vest")


def compliance(helmet: bool, vest: bool) -> Dict[str, object]:
    """Map per-worker PPE presence flags to a compliance verdict.

    Pure, deterministic policy logic - no model is involved. A worker is
    compliant iff *every* required PPE item is present. Whatever is missing is
    returned as a list so the verdict is actionable (who is missing what).

    Parameters
    ----------
    helmet, vest : bool
        Presence flags for each PPE item (already thresholded from Stage 2
        probabilities at the Stage 2 -> Stage 3 boundary).

    Returns
    -------
    dict with keys:
        "compliant"     : bool  - True iff nothing is missing.
        "missing_items" : list  - the required items that are absent.
    """
    present = {"helmet": bool(helmet), "vest": bool(vest)}
    missing: List[str] = [item for item in REQUIRED_PPE if not present[item]]
    return {
        "compliant": len(missing) == 0,
        "missing_items": missing,
    }


# Quick self-checks (act as inline documentation of the contract)
assert compliance(True,  True)  == {"compliant": True,  "missing_items": []}
assert compliance(True,  False) == {"compliant": False, "missing_items": ["vest"]}
assert compliance(False, False) == {"compliant": False, "missing_items": ["helmet", "vest"]}
print("compliance() defined - required PPE:", REQUIRED_PPE)


In [ ]:
# =============================================================================
# Chapter 13.4 - Truth table: every helmet x vest combination
# =============================================================================
import itertools

rows = []
for helmet, vest in itertools.product([True, False], repeat=2):
    verdict = compliance(helmet, vest)
    rows.append({
        "helmet":        helmet,
        "vest":          vest,
        "compliant":     verdict["compliant"],
        "missing_items": ", ".join(verdict["missing_items"]) or "-",
    })

truth_table = pd.DataFrame(rows)
print("Stage 3 - compliance truth table (site policy: helmet AND vest)")
print("=" * 60)
print(truth_table.to_string(index=False))


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.65;">

<h3 style="color: #1e40af; margin-top: 18px;">13.5 Summary</h3>
<p style="text-align: justify;">
Stage&nbsp;3 is intentionally trivial &mdash; and that is the point. All the difficulty lives in the
two learned stages; the final verdict is a transparent <code>AND</code> over their outputs, with a
list of missing items for actionability. The truth table above enumerates the complete behaviour:
only the <code>helmet&nbsp;=&nbsp;True,&nbsp;vest&nbsp;=&nbsp;True</code> row is compliant, and every
other row names exactly what is missing.
</p>
<p style="text-align: justify;">
In <strong>Chapter&nbsp;14</strong> this function becomes the last link of the end-to-end pipeline:
for every person box that Stage&nbsp;1 detects, we crop, run Stage&nbsp;2, threshold its two
probabilities to booleans, and call <code>compliance()</code> to colour the worker's box green
(compliant) or red (missing PPE).
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">14. End-to-End pipeline evaluation</h2>

<p style="text-align: justify;">
Chapters 8&ndash;13 built and evaluated each stage <em>in isolation</em>. This chapter finally runs the
three stages as <strong>one system</strong> on the held-out test images and scores the system as a whole:
</p>

<p style="text-align: center; font-size: 14px;">
image &rarr; <strong>Stage&nbsp;1</strong> (YOLOv8n) &rarr; person boxes &rarr;
<strong>Stage&nbsp;2</strong> (ResNet50 FT, per crop) &rarr; helmet/vest probs &rarr;
threshold@0.5 &rarr; <strong>Stage&nbsp;3</strong> <code>compliance()</code> &rarr; per-worker verdict
</p>

<h3 style="color: #1e40af; margin-top: 18px;">14.1 How the system is scored</h3>
<p style="text-align: justify;">
For every test image we run the detector, classify each detected person, and produce a compliance
verdict. We then <strong>greedily match</strong> the predicted person boxes to the ground-truth person
boxes (IoU&nbsp;&ge;&nbsp;0.5, highest-confidence first). On the <em>matched</em> workers we ask the
end-to-end question: <em>did the full pipeline get the compliance verdict right?</em> The ground-truth
helmet/vest of each person is derived with the exact same Strategy&nbsp;A rule used to build the crop
dataset in Chapter&nbsp;10.
</p>
<p style="text-align: justify;">
We report <strong>detection recall</strong> (what fraction of real workers the detector found) alongside
the compliance accuracy, so the Stage&nbsp;1 and Stage&nbsp;2 contributions stay visible as
<em>layered</em> rather than hidden inside a single number. Operating points: YOLO confidence
<code>0.25</code>, helmet/vest threshold <code>0.5</code>.
</p>

<p style="text-align: justify; font-size: 13px; color: #475569;">
The run itself executed in <code>end_to_end_colab.ipynb</code> (Colab GPU &mdash; the only environment
with both PyTorch-GPU for YOLO and TensorFlow for ResNet50). The cells below load the artifacts it wrote
to <code>results/pipeline/</code>.
</p>

</div>

In [ ]:
# =============================================================================
# Chapter 14.2 - Load the end-to-end evaluation summary
# =============================================================================
PIPELINE_DIR = REPO_ROOT / "results" / "pipeline"

e2e_summary = pd.read_csv(PIPELINE_DIR / "summary.csv")

formatters = {
    "Images":         lambda v: f"{int(v):,}",
    "GT persons":     lambda v: f"{int(v):,}",
    "Pred persons":   lambda v: f"{int(v):,}",
    "Matched":        lambda v: f"{int(v):,}",
    "Det recall":     lambda v: f"{v:.3f}",
    "Det precision":  lambda v: f"{v:.3f}",
    "Compliance acc": lambda v: f"{v:.3f}",
    "Helmet acc":     lambda v: f"{v:.3f}",
    "Vest acc":       lambda v: f"{v:.3f}",
}
print("End-to-End pipeline - YOLOv8n -> ResNet50(FT) -> compliance, scored on matched workers")
print("=" * 110)
print(e2e_summary.to_string(index=False, formatters=formatters))


In [ ]:
# =============================================================================
# Chapter 14.3 - Annotated samples (green = compliant, red = missing PPE)
# =============================================================================
import matplotlib.image as mpimg

for fname, caption in [
    ("samples_test_in_domain.png",     "Test In-Domain (Kaggle CSS test)"),
    ("samples_test_out_of_domain.png", "Test Out-of-Domain (Ultralytics)"),
]:
    img = mpimg.imread(PIPELINE_DIR / fname)
    fig, ax = plt.subplots(figsize=(13, 8))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"End-to-End annotated samples - {caption}\n"
                 f"(green box = compliant, red box = missing PPE)", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
# =============================================================================
# Chapter 14.4 - Compliance confusion matrices (on matched workers)
# =============================================================================
conf_img = mpimg.imread(PIPELINE_DIR / "confusion_matrices.png")
fig, ax = plt.subplots(figsize=(13, 5))
ax.imshow(conf_img)
ax.axis("off")
plt.tight_layout()
plt.show()


<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">14.5 Findings</h3>

<h4 style="color:#1e40af;">14.5.1 In-domain, the full system works very well</h4>
<p style="text-align: justify;">
On the Kaggle CSS test set the detector finds <strong>81%</strong> of the real workers (recall 0.814),
and of those, the full pipeline returns the correct compliance verdict <strong>96.4%</strong> of the time
(135 of 140 matched workers). The confusion matrix shows only <strong>5 errors</strong> in total: 4 workers
wrongly cleared and 1 false alarm. This confirms that the three stages, chained together, behave as a
coherent system in the domain they were tuned on.
</p>

<h4 style="color:#1e40af;">14.5.2 Out-of-domain, errors compound across stages</h4>
<p style="text-align: justify;">
On Ultralytics the system degrades on <em>both</em> axes: detection recall falls to <strong>0.708</strong>
(more workers missed entirely), and end-to-end compliance accuracy drops to <strong>0.662</strong>. That
~30&nbsp;pp end-to-end gap is <em>larger</em> than the gap of any single stage measured alone (Stage&nbsp;1
mAP gap ~20&nbsp;pp, Stage&nbsp;2 AUC gap ~7&nbsp;pp), precisely because the end-to-end number
<strong>stacks both stages' errors</strong>: a worker is scored correctly only if Stage&nbsp;1 localises
them <em>and</em> Stage&nbsp;2 reads their PPE correctly. Composition is multiplicative, not additive.
</p>

<h4 style="color:#1e40af;">14.5.3 The errors are mostly "safe" ones &mdash; and that is by design of the failure mode</h4>
<p style="text-align: justify;">
The out-of-domain confusion matrix is strongly asymmetric. The dominant error is <strong>469 false alarms
(29.5%)</strong> &mdash; workers who <em>are</em> wearing their PPE but are flagged as non-compliant &mdash;
versus only <strong>67 dangerous misses (4.2%)</strong>, where someone missing PPE is wrongly cleared.
This is the end-to-end signature of the Stage&nbsp;2 finding from Chapter&nbsp;12: under the
<strong>class-prior shift</strong> (Ultralytics has far more PPE-positive workers than the training data),
the classifier under-predicts "PPE present", so it over-flags. From a safety standpoint this is the
<em>right direction to fail</em>: a false alarm costs a second glance, while a missed un-helmeted worker
could cost an injury. The system stays conservative even where it is least certain.
</p>

<h4 style="color:#1e40af;">14.5.4 Detection precision and the operating point</h4>
<p style="text-align: justify;">
At the chosen confidence of 0.25 the detector over-produces boxes &mdash; 3,437 predictions for 2,244
real workers out-of-domain (detection precision 0.462). Many are low-confidence or duplicate boxes that
fail to match a ground-truth person. This is a tunable trade-off: raising the YOLO confidence threshold
would lift precision at the cost of recall. We kept a low threshold so the compliance evaluation is run on
as many genuinely-detected workers as possible.
</p>

<h4 style="color:#1e40af;">14.5.5 Bottom line</h4>
<p style="text-align: justify;">
The two-stage design pays off: in-domain the pipeline is reliable (96% compliance accuracy on detected
workers), and even on a completely unseen domain it remains useful (66%) while failing safely toward
over-flagging rather than missing violations. The biggest lever for improvement is Stage&nbsp;1 recall and
Stage&nbsp;2's class-prior robustness on unseen domains &mdash; both discussed as future work in
Chapter&nbsp;15.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">15. Discussion</h2>

<p style="text-align: justify;">
The single thread running through this project is the <strong>Cross-Dataset protocol</strong>: every model
is measured both on the domain it was trained on (Kaggle CSS) and on a domain it has never seen
(Ultralytics). The size of the drop between the two is the empirical answer to the only question that
matters for a safety system meant to be deployed on a <em>new</em> site: <em>does it still work where it
was not trained?</em>
</p>

<h3 style="color: #1e40af; margin-top: 18px;">15.1 Why a Cross-Dataset Gap exists at all</h3>
<p style="text-align: justify;">
Two distinct distribution shifts act at once:
</p>
<ul>
  <li><strong>Visual domain shift.</strong> The two datasets differ in cameras, resolution, scene layout,
      lighting and annotation style. A detector that learned what a "construction worker against this kind
      of background" looks like sees something genuinely different on the new domain.</li>
  <li><strong>Class-prior shift.</strong> In the training crops only ~33% of workers wear a helmet, while in
      Ultralytics ~69% do. A model trained where "the default is no PPE" carries that prior into a world
      where the default is reversed, and systematically under-predicts the positive class.</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">15.2 Why Stage 2 generalises better than Stage 1</h3>
<p style="text-align: justify;">
The measured gaps are strikingly different: <strong>~20&nbsp;pp mAP for Stage&nbsp;1 vs ~7&nbsp;pp AUC for
Stage&nbsp;2</strong> &mdash; the classifier loses only about one third of what the detector loses. The
reason is what each stage actually sees. Stage&nbsp;1 must <em>localise</em> a person against the entire
unseen scene, so the background shift hits it directly. Stage&nbsp;2 receives an already-cropped person:
its input is dominated by the <em>foreground</em> &mdash; a human, with or without a coloured helmet and a
high-visibility vest &mdash; and a human in PPE looks far more alike across domains than the surrounding
construction site does. The residual ImageNet features for "helmet colour" and "vest texture" transfer
well; localisation against a new background does not.
</p>

<h3 style="color: #1e40af; margin-top: 18px;">15.3 What the end-to-end evaluation adds</h3>
<p style="text-align: justify;">
Chaining the stages exposes two things that no single-stage number reveals:
</p>
<ul>
  <li><strong>Errors compound.</strong> The end-to-end compliance gap (~30&nbsp;pp) is <em>larger</em> than
      either stage alone, because a worker is scored correctly only if Stage&nbsp;1 finds them
      <em>and</em> Stage&nbsp;2 reads their PPE correctly. Composition multiplies the two error rates rather
      than averaging them.</li>
  <li><strong>The system fails safe.</strong> Out-of-domain the dominant error is <em>over-flagging</em>
      (469 compliant workers wrongly marked non-compliant) rather than <em>under-flagging</em> (only 67
      missing-PPE workers wrongly cleared). This is the direct downstream consequence of the
      class-prior-driven recall drop in Stage&nbsp;2: under-predicting "PPE present" pushes the verdict
      toward "non-compliant". For a safety application this is the preferable direction to be wrong in.</li>
</ul>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h3 style="color: #1e40af; margin-top: 18px;">15.4 Limitations</h3>
<ul>
  <li><strong>Small validation set.</strong> Stage&nbsp;2 was selected on only ~150 val crops, so the val
      metrics are noisy; the test-in-domain set (155 crops) occasionally beating val is statistical
      variation, not a real effect.</li>
  <li><strong>Strategy&nbsp;A label noise.</strong> A crop is labelled <code>helmet=0</code> whenever no
      Hardhat box is found near the person &mdash; which conflates a genuine bare head with a person who was
      simply not annotated for PPE. This injects some label noise into both training and the end-to-end
      ground truth.</li>
  <li><strong>Single random seed.</strong> Every number is a point estimate from one training run; we do not
      report <code>mean&nbsp;&plusmn;&nbsp;std</code> over seeds, so small differences should not be
      over-interpreted.</li>
  <li><strong>Untuned operating point.</strong> The end-to-end run used YOLO confidence 0.25, which
      over-produces boxes (detection precision 0.46 out-of-domain). The thresholds were fixed, not tuned per
      deployment.</li>
  <li><strong>Compliance accuracy is conditional on detection.</strong> It is measured on workers the
      detector actually found; workers missed by Stage&nbsp;1 are excluded from the compliance score (their
      effect is captured separately by detection recall).</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">15.5 Future work</h3>
<ul>
  <li><strong>Domain adaptation.</strong> Fine-tuning the detector on even a small labelled sample from a
      target site should close most of the Stage&nbsp;1 gap, which is the pipeline's weakest link.</li>
  <li><strong>Counter the class-prior shift.</strong> Re-weighting the loss, or adjusting the decision
      threshold to the target prior, would directly reduce the out-of-domain false-alarm rate.</li>
  <li><strong>More PPE classes.</strong> The compliance logic is already list-driven
      (<code>REQUIRED_PPE</code>), so adding gloves, boots or goggles is a data problem, not an architecture
      change.</li>
  <li><strong>Multi-seed reporting.</strong> Repeating training over several seeds would turn the point
      estimates into <code>mean&nbsp;&plusmn;&nbsp;std</code> and make the gaps statistically defensible.</li>
  <li><strong>Stronger augmentation.</strong> More aggressive colour / lighting augmentation during
      Stage&nbsp;2 training could further harden the helmet/vest features against unseen domains.</li>
</ul>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">16. Summary &amp; conclusions</h2>

<p style="text-align: justify;">
We built a <strong>two-stage PPE-compliance pipeline</strong> &mdash; a YOLOv8n person detector
(Stage&nbsp;1), a fine-tuned ResNet50 multi-label helmet/vest classifier on the detected crops
(Stage&nbsp;2), and a deterministic compliance rule (Stage&nbsp;3) &mdash; and evaluated every component
under a deliberate <strong>Cross-Dataset protocol</strong> (train on Kaggle CSS, test both in-domain and on
the entirely unseen Ultralytics domain). The two-stage decomposition is the project's central design choice,
and the numbers below justify it: by handing the classifier a focused person crop instead of a full scene,
Stage&nbsp;2 generalises roughly three times better than Stage&nbsp;1.
</p>

<table style="border-collapse: collapse; font-size: 13.5px; margin: 10px 0;">
  <thead><tr>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Stage</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Headline metric</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">In-Domain</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Out-of-Domain</th>
    <th style="padding: 6px 12px; border: 1px solid #cbd5e1; background: #f1f5f9; text-align: left;">Cross-Dataset Gap</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Stage 1 &mdash; YOLOv8n (person detection)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">mAP@0.5</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.816</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.617</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>~20 pp</strong></td>
    </tr>
    <tr style="background:#f0fdf4;">
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Stage 2 &mdash; ResNet50 fine-tuned (helmet/vest)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">AUC</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.979</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.912</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>~7 pp</strong></td>
    </tr>
    <tr>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">End-to-End &mdash; full pipeline</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">Compliance acc (matched workers)</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.964</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;">0.662</td>
      <td style="padding: 6px 12px; border: 1px solid #cbd5e1;"><strong>~30 pp</strong></td>
    </tr>
  </tbody>
</table>

<p style="text-align: justify;">
<strong>Conclusions.</strong> (1) In its trained domain the system is reliable &mdash; 96% of detected
workers receive the correct compliance verdict. (2) On a completely unseen domain it remains useful (66%)
and, crucially, <em>fails safe</em>: its errors are dominated by over-flagging compliant workers rather than
clearing unsafe ones. (3) The pipeline's weakest link under domain shift is Stage&nbsp;1 detection, not the
PPE classifier.
</p>

<p style="text-align: justify;">
<strong>Recommendations.</strong> Deploy as-is within the trained domain. For a new site, collect a small
labelled sample and fine-tune the detector (domain adaptation) before relying on the numbers, and tune the
YOLO confidence and PPE thresholds to the site's tolerance for false alarms versus missed violations.
</p>

</div>

<div style="font-family: 'Segoe UI', Arial, sans-serif; line-height: 1.6;">

<h2 style="color: #1e3a8a; border-bottom: 2px solid #cbd5e1; padding-bottom: 4px;">17. References</h2>

<h3 style="color: #1e40af; margin-top: 18px;">Datasets</h3>
<ul>
  <li><strong>Ultralytics Construction-PPE</strong> dataset (out-of-domain test set) &mdash;
      <code>construction-ppe</code>, distributed with the Ultralytics package.</li>
  <li><strong>Construction Site Safety (CSS)</strong> dataset, Roboflow / Kaggle (training and in-domain
      test set).</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">Models &amp; papers</h3>
<ul>
  <li>Jocher, G. et al. <em>YOLOv8</em>. Ultralytics, 2023. (Stage&nbsp;1 detector.)</li>
  <li>Simonyan, K. &amp; Zisserman, A. <em>Very Deep Convolutional Networks for Large-Scale Image
      Recognition</em> (VGG16). ICLR 2015 (arXiv:1409.1556). (Stage&nbsp;2 ablation baseline.)</li>
  <li>He, K., Zhang, X., Ren, S. &amp; Sun, J. <em>Deep Residual Learning for Image Recognition</em>
      (ResNet). CVPR 2016 (arXiv:1512.03385). (Stage&nbsp;2 backbone.)</li>
  <li>Deng, J. et al. <em>ImageNet: A Large-Scale Hierarchical Image Database</em>. CVPR 2009.
      (Source of the pre-trained backbone weights used for transfer learning.)</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">Frameworks &amp; libraries</h3>
<ul>
  <li><strong>PyTorch</strong> &mdash; backend for YOLOv8 training and inference.</li>
  <li><strong>TensorFlow / Keras</strong> &mdash; Stage&nbsp;2 classifier training, fine-tuning and inference.</li>
  <li><strong>Ultralytics</strong> &mdash; YOLOv8 implementation and training API.</li>
  <li><strong>OpenCV</strong>, <strong>NumPy</strong>, <strong>pandas</strong>, <strong>Matplotlib</strong>
      &mdash; data preparation, cropping, analysis and plotting.</li>
</ul>

<h3 style="color: #1e40af; margin-top: 18px;">Course material</h3>
<ul>
  <li>Deep Learning Workshop &mdash; <strong>Unit&nbsp;8: Transfer Learning</strong> (Cats vs Dogs,
      feature extraction &rarr; fine-tuning). The Stage&nbsp;2 methodology follows this unit directly.</li>
</ul>

</div>